# Sóstrato el Arquitecto · Banco de walk-forward

Compara, con las mismas reglas y solo con la información disponible antes de cada período, los modelos del mapa de ocupación contra el **naive** (el mapa real del mismo semestre del año anterior). Responde a la validación independiente del 25-10 (`Documentación/INFORME_VALIDACION_EL_EXPERTO_2026-10-25.md`, hallazgos H1–H3).

**Diseño aprobado por Diego (25-10):**
- **A:** orígenes de pregrado 202520, 202610 y 202620 para decidir; los de AV (202525, 202615, 202625) son informativos. **Ampliado el 26-10:** se suman 202410 y 202420 (deciden) y 202415 y 202425 (AV, informativos). 202625 se marca como foto con la inscripción abierta.
- **B:** la prueba punta a punta es completa solo en 202620 y 202625. En 202520 y 202610 se hace con la grilla conocida (optimista).
- **C:** un tercer grano, unidad × nivel × bloque, es informativo.

| Modelo | Qué es |
|---|---|
| **N0** | Naive: el mapa real del mismo semestre del año anterior. Es la vara (MASE = 1) |
| N1 | Naive ponderado: 2/3 del año anterior + 1/3 del de antes |
| **P1** | Perfil actual (D9 + D11) con los NRC reales del objetivo: error de la etapa 3 aislado |
| **E1** | Punta a punta actual: Euribia → secciones D2 + D8 (regla recalibrada solo con datos anteriores) → perfil → mapa |
| E1s | Punta a punta con secciones sin la fuga de H3: `TEORICAS_PROYECTADAS` × factor calibrado con datos anteriores |
| G1 / G1b | Programación sintética por estrato (D13): secciones físicas anónimas remuestreadas del año anterior hasta el volumen de inscritos; G1b con elasticidad salas/inscritos |
| **C2** | Combinación w·N0 + (1−w)·H2 (hábito de sala con el estándar real), w calibrado igual que C1 |
| **C1** | Combinación w·N0 + (1−w)·G1b. w (de 0 a 1, en pasos de 0,1) se elige solo con los orígenes **anteriores** del mismo nivel. En el primer origen de cada nivel no hay con qué calibrar y w = 1 (= naive) |

**Regla de adopción:** un modelo reemplaza al naive solo con **MASE < 1 en la mayoría de los orígenes de pregrado** (con 5 orígenes, al menos 3; el 26-10 se sumaron 202410 y 202420 a los 3 iniciales), en los dos granos para decidir (celda y campus × día × bloque).

**Información permitida en cada origen** (T10 §3.3): todo parámetro se calcula con períodos anteriores al objetivo. Eso incluye la ventana del perfil, el factor de compartición, la regla de secciones y la clasificación de espacios, que para predecir solo usa salas vistas antes del objetivo. La **verdad** (el mapa real del objetivo) se clasifica con todo lo conocido después.

## 0 · Configuración y carga

In [61]:
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if p.name.startswith("Sóstrato el Arquitecto"))
REPO_DP = RAIZ.parent
BBDD = RAIZ / "BBDD"
DESCARGAS = Path.home() / "Downloads"
EURIBIA = DESCARGAS / "Proyección Demanda Asiganturas BBDD"

ORIGENES = {202410: "decide", 202420: "decide", 202520: "decide", 202610: "decide", 202620: "decide",
            202415: "informativo", 202425: "informativo", 202525: "informativo", 202615: "informativo",
            202625: "informativo (foto con inscripción abierta)"}
PERIODOS_PRONOSTICO_EURIBIA = {202620, 202625}          # el resto: grilla conocida (optimista)
N_SIM = 100
SEMILLA = 2049
CLASES = ["SALA", "LABORATORIO"]
DIAS = ["LUNES", "MARTES", "MIERCOLES", "JUEVES", "VIERNES", "SABADO", "DOMINGO"]
NB = 20
MIN_NRC_PERFIL = 3
PESO_ANIO = {1: 2 / 3, 2: 1 / 3}
MIN_FILAS_FACTOR = 30
CANDIDATAS = ["N_SECCIONES_TEORICAS_PROYECTADAS", "N_SECCIONES_P25", "N_SECCIONES_P50",
              "N_SECCIONES_P75", "N_SECCIONES_P90"]
CLASE_POR_COMPONENTE = {"TEO": "SALA", "LAB": "LABORATORIO", "SIM": "LABORATORIO", "SSA": "LABORATORIO"}
NIVELES = [("SERIE", ["COD_SERIE", "COD_COMPONENTE", "COD_CAMPUS"]),
           ("SERIE_OTRO_CAMPUS", ["COD_SERIE", "COD_COMPONENTE"]),
           ("MATERIA", ["COD_MATERIA", "COD_COMPONENTE", "COD_CAMPUS"]),
           ("COMPONENTE_CAMPUS", ["COD_COMPONENTE", "COD_CAMPUS"]),
           ("COMPONENTE", ["COD_COMPONENTE"])]
NIVEL_FIJA_CAMPUS = {n: "COD_CAMPUS" in k for n, k in NIVELES}
tipo_periodo = lambda p: np.where(np.isin(np.asarray(p) % 100, [10, 20]), "UG 10/20", "AV 05/15/25")

# Salidas de PIPELINES.ipynb que no dependen del período a predecir
ph = pd.read_parquet(BBDD / "PROG_HIST_NRC.parquet")
pl = pd.read_parquet(BBDD / "PLANTILLAS_NRC.parquet",
                     columns=["PERIODO", "NRC", "COD_COMPONENTE", "DIA", "BLOQUE", "COD_CAMPUS_EDIFICIO",
                              "EDIFICIO", "SALA", "CLASE_ESPACIO", "PESO_SEMANAS", "PESO_LC"])
pl["CON_SALA"] = pl["CLASE_ESPACIO"] != "SIN SALA"
unidad = pd.read_parquet(BBDD / "UNIDAD_ACADEMICA.parquet")
pf = pd.read_csv(REPO_DP / "Heat-Map" / "BBDD" / "PLANTA_FISICA_PREPROCESADA.csv", sep=None, engine="python")
pf = pf[pf["Es_Mas_Actualizado"] == 1].copy()
pf["EDIFICIO"] = pf["Código Edificio"].astype(str); pf["SALA"] = pf["Número Sala"].astype(str)
planta = pf.drop_duplicates(["EDIFICIO", "SALA"]).set_index(["EDIFICIO", "SALA"])["CLASIFICACIÓN"]
moda_edificio = pf.groupby("EDIFICIO")["CLASIFICACIÓN"].agg(lambda s: s.mode().iat[0])

leer = lambda f: pd.read_csv(EURIBIA / f, sep=";", decimal=",", dtype={"CURSO": str, "PERIODO": int}, low_memory=False)
eur_proy = leer("PROYECCION_INSCRITOS_2026_2027.csv")
eur_serie = leer("SERIE_COMPLETA_REAL_VS_PROYECTADO.csv")

atrib = ph[["PERIODO", "NRC", "COD_ASIGNATURA", "COD_ASIGNATURA_EURIBIA", "COD_MATERIA", "COD_COMPONENTE", "COD_CAMPUS"]].copy()
atrib["COD_SERIE"] = atrib["COD_ASIGNATURA_EURIBIA"].fillna(atrib["COD_ASIGNATURA"])

# Codificación de celdas
CAMP = {c: i for i, c in enumerate(sorted(set(pl["COD_CAMPUS_EDIFICIO"].dropna()) | set(ph["COD_CAMPUS"].dropna())))}
DIA_I = {d: i for i, d in enumerate(DIAS)}
CLA_I = {c: i for i, c in enumerate(CLASES)}
UNI = {u: i for i, u in enumerate(sorted(unidad["DESC_UNIDAD_ACADEMICA"].unique()))}
UNI_MATERIA = unidad.set_index("COD_MATERIA")["DESC_UNIDAD_ACADEMICA"].map(UNI)
N_SC = len(DIAS) * NB * len(CLASES)
N_CELDAS = len(CAMP) * N_SC
N_UNI = len(UNI) * NB
resto = lambda d, b, k: (d * NB + b) * len(CLASES) + k
print(f"NRC: {len(ph):,} · filas de plantilla: {len(pl):,} · campus: {len(CAMP)} · unidades: {len(UNI)}".replace(",", "."))

NRC: 170.452 · filas de plantilla: 2.236.891 · campus: 9 · unidades: 63


## 1 · Piezas que dependen del origen

- **`clasificar(periodos)`**: la clase de cada sala para el mapa. Las salas de la planta toman su clase; las que se usaron en `periodos` y no están en la planta heredan la moda de su edificio o, si el edificio no está, se deciden por componente. Lo demás queda fuera del mapa. Es la misma regla de D10, pero limitada a los períodos indicados.
- **`mapa_real(T)`**: la verdad. Cada sala cuenta como mucho 1 por bloque; para el grano de unidad, ese valor se reparte entre los NRC de la sala según su peso.
- **`factor_comparticion(T)`**: salas reales ÷ Σ `PESO_OCUPACION` por campus × clase, con los períodos de los 2 años anteriores al objetivo (D11).

In [62]:
def clasificar(periodos):
    usadas = pl[pl["PERIODO"].isin(periodos) & pl["CON_SALA"]]
    fuera = usadas[~usadas.set_index(["EDIFICIO", "SALA"]).index.isin(planta.index)]
    comp = fuera.groupby(["EDIFICIO", "SALA"])["COD_COMPONENTE"].agg(lambda s: s.mode().iat[0]).reset_index()
    comp["CLASE"] = comp["EDIFICIO"].map(moda_edificio).fillna(comp["COD_COMPONENTE"].map(CLASE_POR_COMPONENTE))
    clase = pd.concat([planta, comp.set_index(["EDIFICIO", "SALA"])["CLASE"]])
    return clase.where(clase.isin(CLASES), "FUERA")


def filas_clasificadas(periodos_filas, clase):
    x = pl[pl["PERIODO"].isin(periodos_filas) & pl["CON_SALA"]].copy()
    x["CLASE"] = clase.reindex(pd.MultiIndex.from_frame(x[["EDIFICIO", "SALA"]])).to_numpy()
    return x[x["CLASE"].isin(CLASES)]


def _celda(df, campus_col="COD_CAMPUS_EDIFICIO"):
    return (df[campus_col].map(CAMP).to_numpy() * N_SC
            + resto(df["DIA"].map(DIA_I).to_numpy(), df["BLOQUE"].to_numpy(), df["CLASE"].map(CLA_I).to_numpy()))


def mapa_real(T):
    # Verdad del período T, clasificada con todo lo conocido (incluido T)
    x = filas_clasificadas([T], clasificar([p for p in pl["PERIODO"].unique() if T - 200 <= p <= T]))
    llave = ["EDIFICIO", "SALA", "DIA", "BLOQUE"]
    suma = x.groupby(llave)["PESO_SEMANAS"].transform("sum")
    x["OCUPADA"] = x["PESO_SEMANAS"] / suma * suma.clip(upper=1)            # reparte la sala ocupada entre sus NRC
    celda = np.bincount(_celda(x), weights=x["OCUPADA"].to_numpy(), minlength=N_CELDAS)
    x = x.merge(ph[["PERIODO", "NRC", "COD_MATERIA"]], on=["PERIODO", "NRC"])
    u = x["COD_MATERIA"].map(UNI_MATERIA).to_numpy()
    uni = np.bincount(u * NB + x["BLOQUE"].to_numpy(), weights=x["OCUPADA"].to_numpy(), minlength=N_UNI)
    return celda, uni


def factor_comparticion(T):
    previos = [p for p in pl["PERIODO"].unique() if T - 200 < p < T]
    x = filas_clasificadas(previos, clasificar(previos))
    salas = x.groupby(["PERIODO", "EDIFICIO", "SALA", "DIA", "BLOQUE"]).agg(
        O=("PESO_SEMANAS", "sum"), C=("COD_CAMPUS_EDIFICIO", "first"), K=("CLASE", "first"))
    salas["O"] = salas["O"].clip(upper=1)
    num = salas.groupby(["C", "K"])["O"].sum()
    den = (x["PESO_SEMANAS"] * x["PESO_LC"]).groupby([x["COD_CAMPUS_EDIFICIO"], x["CLASE"]]).sum()
    f = (num / den.rename_axis(["C", "K"]))
    por_clase = salas.groupby("K")["O"].sum() / (x["PESO_SEMANAS"] * x["PESO_LC"]).groupby(x["CLASE"]).sum()
    return f, por_clase

## 2 · Simulación del perfil (D9 + D11), sin información del objetivo

Cada sección de destino toma una plantilla de la ventana (T−100 con 2/3, T−200 con 1/3) por la cascada. Las plantillas se clasifican con las salas vistas **antes** de T. Cada fila pesa `PESO_SEMANAS` × `PESO_LC` × el factor de compartición de T.

In [63]:
def asignar_pool(destino, base):
    destino = destino.copy(); destino["NIVEL"] = None; destino["POOL"] = None
    miembros = []
    for nivel, llave in NIVELES:
        conteo = base.groupby(llave).size().rename("N")
        sin = destino["NIVEL"].isna()
        cand = destino.loc[sin, llave].merge(conteo, left_on=llave, right_index=True, how="left")["N"].fillna(0)
        idx = destino.index[sin][cand.to_numpy() >= MIN_NRC_PERFIL]
        if len(idx) == 0:
            continue
        destino.loc[idx, "NIVEL"] = nivel
        destino.loc[idx, "POOL"] = nivel + "|" + destino.loc[idx, llave].astype(str).agg("|".join, axis=1)
        m = base.merge(destino.loc[idx, llave].drop_duplicates(), on=llave)
        m["POOL"] = nivel + "|" + m[llave].astype(str).agg("|".join, axis=1)
        miembros.append(m[["POOL", "PERIODO", "NRC", "PESO_VENTANA"]])
    return destino, pd.concat(miembros, ignore_index=True)


def simular(destino, T, n_sim=N_SIM):
    # destino: una fila por sección con COD_SERIE, COD_MATERIA, COD_COMPONENTE, COD_CAMPUS
    pesos = {T - 100 * a: w for a, w in PESO_ANIO.items()}
    base = atrib[atrib["PERIODO"].isin(pesos)].copy()
    base["PESO_VENTANA"] = base["PERIODO"].map(pesos)
    destino, miembros = asignar_pool(destino.reset_index(drop=True), base)
    sin_horario = destino["POOL"].isna().mean()
    destino = destino[destino["POOL"].notna()].reset_index(drop=True)

    previos = [p for p in pl["PERIODO"].unique() if T - 200 <= p < T]
    fis = filas_clasificadas(list(pesos), clasificar(previos))
    f, f_clase = factor_comparticion(T)
    fk = pd.Series(list(zip(fis["COD_CAMPUS_EDIFICIO"], fis["CLASE"])), index=fis.index).map(f.to_dict())
    fk = fk.fillna(fis["CLASE"].map(f_clase)).fillna(1.0)
    fis = fis.assign(PESO=fis["PESO_SEMANAS"] * fis["PESO_LC"] * fk)

    ids = miembros[["PERIODO", "NRC"]].drop_duplicates().reset_index(drop=True); ids["TID"] = np.arange(len(ids))
    cel = fis.merge(ids, on=["PERIODO", "NRC"]).sort_values("TID")
    cel_resto = resto(cel["DIA"].map(DIA_I).to_numpy(), cel["BLOQUE"].to_numpy(), cel["CLASE"].map(CLA_I).to_numpy())
    cel_camp, cel_peso, cel_bloque = cel["COD_CAMPUS_EDIFICIO"].map(CAMP).to_numpy(), cel["PESO"].to_numpy(), cel["BLOQUE"].to_numpy()
    ini = np.searchsorted(cel["TID"].to_numpy(), ids["TID"].to_numpy(), "left")
    fin = np.searchsorted(cel["TID"].to_numpy(), ids["TID"].to_numpy(), "right")

    m = miembros.merge(ids, on=["PERIODO", "NRC"])
    pid = {p: i for i, p in enumerate(m["POOL"].unique())}
    m["PID"] = m["POOL"].map(pid); m = m.sort_values(["PID", "TID"])
    m["ACUM"] = m.groupby("PID")["PESO_VENTANA"].cumsum() / m.groupby("PID")["PESO_VENTANA"].transform("sum")
    clave, tid_m = m["PID"].to_numpy() + m["ACUM"].to_numpy() - 1e-12, m["TID"].to_numpy()
    d_pid = destino["POOL"].map(pid).to_numpy()
    d_camp = destino["COD_CAMPUS"].map(CAMP).fillna(0).astype(int).to_numpy()
    d_fija = destino["NIVEL"].map(NIVEL_FIJA_CAMPUS).to_numpy()
    d_uni = destino["COD_MATERIA"].map(UNI_MATERIA).fillna(-1).astype(int).to_numpy()

    rng = np.random.default_rng(SEMILLA)
    s_celda, s_uni = np.zeros((n_sim, N_CELDAS)), np.zeros((n_sim, N_UNI))
    for s in range(n_sim):
        t = tid_m[np.searchsorted(clave, d_pid + rng.random(len(d_pid)))]
        largo = fin[t] - ini[t]
        filas = np.repeat(ini[t], largo) + (np.arange(largo.sum()) - np.repeat(np.cumsum(largo) - largo, largo))
        camp = np.where(np.repeat(d_fija, largo), cel_camp[filas], np.repeat(d_camp, largo))
        s_celda[s] = np.bincount(camp * N_SC + cel_resto[filas], weights=cel_peso[filas], minlength=N_CELDAS)
        u = np.repeat(d_uni, largo); ok = u >= 0
        s_uni[s] = np.bincount(u[ok] * NB + cel_bloque[filas][ok], weights=cel_peso[filas][ok], minlength=N_UNI)
    return s_celda, s_uni, sin_horario

## 3 · Destinos de cada modelo

- **P1:** los NRC reales del objetivo.
- **E1 / E1s:** las secciones que salen de Euribia para el objetivo. En 202620 y 202625 se usa su proyección (`PROYECCION_INSCRITOS_2026_2027.csv`). En los demás orígenes, las filas `REAL` de `SERIE_COMPLETA`: el pronóstico es genuino, porque cada año se entrena con años anteriores, pero la **grilla de asignaturas es la que se dictó** (optimista).
  - **E1:** la regla D2 + D8 recalibrada solo con filas `REAL` anteriores al objetivo. Arrastra la fuga de H3 dentro de esas filas.
  - **E1s:** `TEORICAS_PROYECTADAS` × factor por tipo de período × componente (Σ reales ÷ Σ teóricas, filas anteriores). Es una norma fija, así que no hay fuga.
  - En ambas se excluyen las secciones **OCS**. Euribia proyecta unas 3.000 por período, pero en Cirilo casi no tienen horario. Si quedan, la cascada las amontona en las pocas plantillas OCS de la ventana y deforma el mapa (en 202520, la correlación caía a 0,44). Es una falla del método vigente, que queda anotada en el trabajo de H2.

In [64]:
def wape(r, p):
    return np.abs(p - r).sum() / r.sum()


def filas_euribia(T):
    # El código de serie sigue la misma convención con que PIPELINES §2 armó COD_SERIE en las plantillas:
    # en la proyección, CODIGO_PROGRAMABLE; en la historia (filas REAL), ASIGNATURA_REGLA
    if T in PERIODOS_PRONOSTICO_EURIBIA:
        f = eur_proy[eur_proy["PERIODO"] == T].copy()
        return f.assign(COD_SERIE_EUR=f["CODIGO_PROGRAMABLE"]), "pronóstico"
    f = eur_serie[(eur_serie["TIPO"] == "REAL") & (eur_serie["PERIODO"] == T)].copy()
    return f.assign(COD_SERIE_EUR=f["ASIGNATURA_REGLA"]), "grilla conocida (optimista)"


COMPONENTES_SIN_HORARIO = {"OCS"}     # Euribia proyecta ~3.000 secciones OCS por período; en Cirilo casi no tienen horario


def secciones(T, variante):
    filas, grilla = filas_euribia(T)
    filas = filas[~filas["COMPONENTE"].isin(COMPONENTES_SIN_HORARIO)].copy()
    filas["TIPO_PERIODO"] = tipo_periodo(filas["PERIODO"])
    h = eur_serie[(eur_serie["TIPO"] == "REAL") & (eur_serie["PERIODO"] < T) & eur_serie["N_SECCIONES_REALES"].notna()].copy()
    h["TIPO_PERIODO"] = tipo_periodo(h["PERIODO"])
    seg = ["TIPO_PERIODO", "COMPONENTE"]
    if variante == "D2D8":
        w = h.groupby(seg).apply(lambda g: pd.Series({c: wape(g["N_SECCIONES_REALES"], g[c]) for c in CANDIDATAS}), include_groups=False)
        col = w.idxmin(axis=1).rename("COL")
    else:
        col = pd.Series("N_SECCIONES_TEORICAS_PROYECTADAS", index=h.groupby(seg).size().index, name="COL")
    h = h.join(col, on=seg)
    h["ELEG"] = h.to_numpy()[np.arange(len(h)), h.columns.get_indexer(h["COL"])].astype(float)
    a = h.groupby(seg).agg(N=("ELEG", "size"), R=("N_SECCIONES_REALES", "sum"), E=("ELEG", "sum"))
    f_tipo = h.groupby("TIPO_PERIODO")["N_SECCIONES_REALES"].sum() / h.groupby("TIPO_PERIODO")["ELEG"].sum()
    fac = pd.Series(np.where(a["N"] >= MIN_FILAS_FACTOR, a["R"] / a["E"],
                             a.index.get_level_values(0).map(f_tipo)), index=a.index, name="FAC")
    filas = filas.join(col, on=seg).join(fac, on=seg)
    filas["COL"] = filas["COL"].fillna("N_SECCIONES_P50"); filas["FAC"] = filas["FAC"].fillna(1.0)
    base = filas.to_numpy()[np.arange(len(filas)), filas.columns.get_indexer(filas["COL"])].astype(float)
    filas["N"] = np.maximum(1, np.round(np.nan_to_num(base) * filas["FAC"])).astype(int)
    d = filas.loc[filas.index.repeat(filas["N"]), ["COD_SERIE_EUR", "MATERIA", "COMPONENTE", "CAMPUS"]]
    d.columns = ["COD_SERIE", "COD_MATERIA", "COD_COMPONENTE", "COD_CAMPUS"]
    # Qué parte de la verdad viene de NRC cuya combinación tiene proyección (la compensación de H2)
    reales = atrib[atrib["PERIODO"] == T]
    con = reales.merge(filas[["COD_SERIE_EUR", "COMPONENTE", "CAMPUS"]].drop_duplicates()
                         .rename(columns={"COD_SERIE_EUR": "COD_SERIE", "COMPONENTE": "COD_COMPONENTE", "CAMPUS": "COD_CAMPUS"}),
                       on=["COD_SERIE", "COD_COMPONENTE", "COD_CAMPUS"])[["PERIODO", "NRC"]]
    x = filas_clasificadas([T], clasificar([p for p in pl["PERIODO"].unique() if T - 200 <= p <= T]))
    explicado = x.merge(con, on=["PERIODO", "NRC"])["PESO_SEMANAS"].sum() / x["PESO_SEMANAS"].sum()
    return d.reset_index(drop=True), grilla, int(filas["N"].sum()), explicado

## 4 · Métricas

Por grano: WAPE, sesgo, correlación y coincidencia en las 30 celdas más cargadas. Para los modelos que simulan, además: cobertura P10–P90, ancho medio y **CRPS relativo** (Σ CRPS ÷ Σ real, en la escala del WAPE). En el naive, el CRPS es igual a su error absoluto.

Granos:
- **celda** (campus × día × bloque × clase) y **campus × día × bloque**: para decidir;
- **unidad × bloque**: informativo. El nivel es fijo en cada origen, porque cada uno es de un solo sufijo.

In [65]:
def crps_ensamble(sims, y):
    s = np.sort(sims, axis=0); n = s.shape[0]
    e1 = np.abs(s - y).mean(0)
    e2 = (2 * (2 * np.arange(1, n + 1) - n - 1)[:, None] * s).sum(0) / n ** 2
    return e1 - 0.5 * e2


def agr_cdb(v):
    return v.reshape(*v.shape[:-1], len(CAMP), len(DIAS), NB, len(CLASES)).sum(-1).reshape(*v.shape[:-1], -1)


def metricas(real, pred):
    # pred: vector (punto) o matriz n_sim × celdas (simulación)
    sim = pred.ndim == 2
    media = pred.mean(0) if sim else pred
    act = (real > 0) | (media > 0)
    r, m = real[act], media[act]
    top = 30
    out = {"WAPE": wape(r, m), "SESGO": m.sum() / r.sum() - 1, "CORR": np.corrcoef(m, r)[0, 1],
           "TOP30": len(set(np.argsort(real)[-top:]) & set(np.argsort(media)[-top:])) / top}
    if sim:
        p10, p90 = np.percentile(pred, 10, 0)[act], np.percentile(pred, 90, 0)[act]
        out.update({"COBERTURA_P10_P90": ((r >= p10) & (r <= p90)).mean(), "ANCHO_MEDIO": (p90 - p10).mean(),
                    "CRPS_REL": crps_ensamble(pred[:, act], r).sum() / r.sum()})
    else:
        out.update({"CRPS_REL": np.abs(m - r).sum() / r.sum()})
    return out

## 4b · G1 · Programación sintética por estrato (decisión del 26-10)

La programación sintética **no necesita saber qué asignatura se dicta**. Basta con saber que habrá una clase de tantos inscritos, en tal horario y en tal sala. La unidad es la **sección física**: una fila anónima con campus, edificio, sala, clase, día(s) y bloques, inscritos, unidad académica, nivel y componente.

**Decisiones de Diego (26-10):**
- **Estrato** = campus × unidad académica × nivel × clase de espacio × componente × período académico. La base es el mismo sufijo del año anterior.
- **Volumen** = inscritos.
- **Salas:** cada sección se reasigna a una sala libre equivalente; lo que no cabe es **déficit**.

**Sección física.** Los NRC de una misma LC, y los NRC que comparten una sala en el mismo bloque al mismo tiempo (Σ `PESO_SEMANAS` > 1 en la sala-bloque), se unen en **una sola sección** que suma sus inscritos. Es la definición de LC de D11, extendida a la compartición que no viene codificada. Así no hace falta el factor residual. La ocupación de la sección en cada sala-bloque es min(1, Σ `PESO_SEMANAS`).

**Generación** (por estrato, en cada simulación):
1. La base son las secciones del año anterior (T−100) del estrato. Si el estrato no existía, se usa T−200; si tampoco, el estrato sin unidad (campus × nivel × clase × componente) de T−100, ubicado en el campus propio.
2. Se ordenan copias de la base al azar: primero la base completa en orden aleatorio, después copias extra. Se toman secciones hasta acercarse al **volumen objetivo** de inscritos: max(inscritos, 1) por sección. Si el estrato crece, se repiten secciones; si se achica, se quitan.
3. **G1b · elasticidad salas/inscritos (26-10):** los inscritos crecen más que la ocupación de salas, porque las clases absorben alumnos. En G1b, un estrato crece en salas como (inscritos objetivo ÷ inscritos base)^β. β se estima por mínimos cuadrados ponderados con los pares de períodos (P, P−100) de los 2 años anteriores a T, del mismo nivel, **sin usar T**.
4. **G1 aislado:** el volumen objetivo es el real del estrato en T (oráculo de volumen). Mide si «el año anterior remuestreado al volumen correcto» le gana al naive. G2, con el volumen proyectado, va después.

**Asignación de salas** (una simulación, para informar el déficit): las secciones del año anterior quedan en su sala. Las copias extra buscan una sala libre de la misma clase, con capacidad ≥ sus inscritos, en todos sus bloques: primero en el mismo edificio y después en el mismo campus. Lo que no cabe es déficit. Como la reasignación es dentro del mismo campus y clase, **no cambia las celdas del mapa**: sirve para entregar la programación «en tal sala» y para medir el déficit.

In [66]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

UNIDAD_MAT = unidad.set_index("COD_MATERIA")["DESC_UNIDAD_ACADEMICA"]
oferta = pd.read_parquet(BBDD / "OFERTA_ESPACIOS.parquet")
oferta = oferta[oferta["ACTIVO"] & oferta["CLASE_MAPA"].isin(CLASES)]


def secciones_fisicas(P, clase):
    # Secciones físicas del período P: LC + compartición simultánea de sala, con sus filas colapsadas por sala-bloque
    x = filas_clasificadas([P], clase)
    meta = ph[ph["PERIODO"] == P][["NRC", "COD_LC", "N_INSCRITOS", "COD_MATERIA", "COD_COMPONENTE", "COD_CAMPUS"]]
    x = x.merge(meta.drop(columns="COD_COMPONENTE"), on="NRC")
    x["G0"] = x["COD_LC"].fillna("N" + x["NRC"].astype(str))
    # Unir grupos que comparten sala-bloque a la vez, SOLO si tienen horario prácticamente idéntico
    # (≥ 80 % de los casilleros de la MÁS GRANDE): una cátedra compartida de verdad. Corrección del 26-10: con
    # «cualquier casillero compartido», y después con «80 % de la más chica», la unión era transitiva (una sección
    # de un bloque servía de puente) y encadenaba cientos de secciones de un mismo laboratorio o auditorio.
    cod, uniq = pd.factorize(x["G0"])
    x["_G"] = cod
    llave = ["EDIFICIO", "SALA", "DIA", "BLOQUE"]
    x["_SID"] = x.groupby(llave).ngroup()
    slot = x.groupby("_SID")["PESO_SEMANAS"].transform("sum") > 1 + 1e-6
    n_slots = x.drop_duplicates(["_G", "_SID"]).groupby("_G").size()
    e = x.loc[slot, ["_SID", "_G"]].drop_duplicates()
    pares = e.merge(e, on="_SID")
    pares = pares[pares["_G_x"] < pares["_G_y"]].groupby(["_G_x", "_G_y"]).size().rename("COMUN").reset_index()
    mayor = np.maximum(pares["_G_x"].map(n_slots), pares["_G_y"].map(n_slots))
    pares = pares[pares["COMUN"] >= 0.8 * mayor]
    adj = coo_matrix((np.ones(len(pares)), (pares["_G_x"].to_numpy(), pares["_G_y"].to_numpy())), shape=(len(uniq), len(uniq)))
    _, comp = connected_components(adj, directed=False)
    x["SEC"] = comp[cod]
    x = x.drop(columns=["_G", "_SID"])
    # Atributos de la sección: inscritos sumados por NRC; unidad, componente y campus del NRC con más inscritos; clase dominante
    nrc = x.drop_duplicates(["SEC", "NRC"])
    ins = nrc.groupby("SEC")["N_INSCRITOS"].sum()
    dom = nrc.sort_values("N_INSCRITOS", ascending=False).drop_duplicates("SEC").set_index("SEC")
    clase_dom = x.groupby(["SEC", "CLASE"])["PESO_SEMANAS"].sum().reset_index().sort_values("PESO_SEMANAS").drop_duplicates("SEC", keep="last").set_index("SEC")["CLASE"]
    sec = pd.DataFrame({"INSCRITOS": ins, "COD_CAMPUS": dom["COD_CAMPUS"], "UNIDAD": dom["COD_MATERIA"].map(UNIDAD_MAT).fillna("SIN UNIDAD"),
                        "COMPONENTE": dom["COD_COMPONENTE"], "CLASE": clase_dom, "MATERIA": dom["COD_MATERIA"]})
    sec["VOL"] = sec["INSCRITOS"].clip(lower=1)
    filas = (x.groupby(["SEC", "EDIFICIO", "SALA", "DIA", "BLOQUE", "CLASE"])
              .agg(PESO=("PESO_SEMANAS", "sum"), CAMP=("COD_CAMPUS_EDIFICIO", "first")).reset_index())
    filas["PESO"] = filas["PESO"].clip(upper=1)
    return sec.reset_index(), filas


ESTRATO = ["COD_CAMPUS", "UNIDAD", "CLASE", "COMPONENTE"]          # + nivel y período: fijos en cada origen
ESTRATO_GRUESO = ["COD_CAMPUS", "CLASE", "COMPONENTE"]

In [67]:
def elasticidad(T):
    # β: cuánto crece la ocupación de salas de un estrato cuando crecen sus inscritos, ocup_P/ocup_{P-100} = (insc_P/insc_{P-100})^β.
    # Se calibra solo con pares de períodos anteriores a T (los últimos 2 años del mismo nivel), sin mirar T.
    nivel = TIPO_POR_SUF[T % 100]
    pares = [p for p in sorted(pl["PERIODO"].unique()) if T - 200 <= p < T and TIPO_POR_SUF.get(p % 100) == nivel
             and (p - 100) in set(pl["PERIODO"].unique())]
    clase = clasificar([p for p in pl["PERIODO"].unique() if p < T])
    xs, ys, ws = [], [], []
    for P in pares:
        est = {}
        for Q in (P, P - 100):
            s, f = secciones_fisicas(Q, clase)
            occ = f.groupby("SEC")["PESO"].sum()
            s = s.assign(OCC=s["SEC"].map(occ).fillna(0))
            est[Q] = s.groupby(ESTRATO)[["VOL", "OCC"]].sum()
        j = est[P].join(est[P - 100], lsuffix="_P", rsuffix="_A", how="inner")
        j = j[(j[["VOL_P", "VOL_A", "OCC_P", "OCC_A"]] > 0).all(axis=1)]
        xs.append(np.log(j["VOL_P"] / j["VOL_A"])); ys.append(np.log(j["OCC_P"] / j["OCC_A"])); ws.append(j["OCC_A"])
    x, y, w = np.concatenate(xs), np.concatenate(ys), np.concatenate(ws)
    beta = float(np.clip((w * x * y).sum() / (w * x * x).sum(), 0, 1.5))
    return beta, pares


TIPO_POR_SUF = {10: "UG", 20: "UG", 5: "AV", 15: "AV", 25: "AV"}


def generar_g1(T, n_sim=N_SIM, beta=None, razon=None, base=None, tol=None):
    # tol: tolerancia de capacidad por clase; si se da, guarda también la ocupación por rango de capacidad (§8)
    # base: período cuyas secciones se remuestrean (por defecto T−100; §7 lo usa para proyectar a varios años)
    B = T - 100 if base is None else base
    lo, hi = (T - 200, T) if base is None else (B - 200, B + 1)
    clase_prev = clasificar([p for p in pl["PERIODO"].unique() if lo <= p < hi])
    s1, f1 = secciones_fisicas(B, clase_prev)
    if (B - 100) in set(pl["PERIODO"].unique()):
        s2, f2 = secciones_fisicas(B - 100, clase_prev)
    else:
        s2, f2 = s1.iloc[0:0].copy(), f1.iloc[0:0].copy()
    s2["SEC"] += s1["SEC"].max() + 1; f2["SEC"] += s1["SEC"].max() + 1
    secs, filas = pd.concat([s1.assign(ORIGEN_BASE=1), s2.assign(ORIGEN_BASE=2)]), pd.concat([f1, f2])
    # Volumen objetivo: el real de T por estrato (G1 aislado, oráculo de volumen)
    sT, _ = secciones_fisicas(T, clasificar([p for p in pl["PERIODO"].unique() if T - 200 <= p <= T]))
    obj = sT.groupby(ESTRATO)["VOL"].sum().rename("OBJ").reset_index()
    if razon is not None:
        # §4e · volumen proyectado: las secciones de T−100 escaladas por la razón de su tipo, en vez del real de T
        r1 = s1.merge(razon.reset_index(), on=list(razon.index.names), how="left").fillna({"RAZON": 1.0})
        obj = (r1["VOL"] * r1["RAZON"]).groupby([r1[c] for c in ESTRATO]).sum().rename("OBJ").reset_index()

    # Base de cada estrato objetivo: T−100 → T−200 → estrato grueso de T−100
    b1 = s1.merge(obj[ESTRATO], on=ESTRATO).assign(EST=lambda d: d[ESTRATO].astype(str).agg("|".join, axis=1), PRESTADO=False, EN_BASE=True)
    tiene1 = set(b1["EST"])
    obj["EST"] = obj[ESTRATO].astype(str).agg("|".join, axis=1)
    falt = obj[~obj["EST"].isin(tiene1)]
    b2 = s2.merge(falt[ESTRATO + ["EST"]], on=ESTRATO).assign(PRESTADO=False)
    falt = falt[~falt["EST"].isin(set(b2["EST"]))]
    b3 = s1.merge(falt[ESTRATO_GRUESO + ["EST", "UNIDAD"]].rename(columns={"UNIDAD": "UNIDAD_DESTINO"}), on=ESTRATO_GRUESO).assign(PRESTADO=True)
    base = pd.concat([b1, b2, b3], ignore_index=True)
    sin_base = obj.loc[~obj["EST"].isin(set(base["EST"])), "OBJ"].sum() / obj["OBJ"].sum()
    base = base.merge(obj[["EST", "OBJ", "COD_CAMPUS"]].rename(columns={"COD_CAMPUS": "CAMPUS_DESTINO"}), on="EST")
    if "UNIDAD_DESTINO" in base:
        base["UNIDAD"] = base["UNIDAD_DESTINO"].fillna(base["UNIDAD"])
    vol_base = base.groupby("EST")["VOL"].transform("sum")
    base["RZ"] = base["OBJ"] / vol_base                                  # razón de volumen del estrato (antes de β)
    if beta is not None:
        # G1b: el estrato crece en salas como (inscritos objetivo / inscritos base)^β; los estratos prestados no se ajustan
        ajusta = ~base["PRESTADO"]
        base.loc[ajusta, "OBJ"] = vol_base[ajusta] * (base.loc[ajusta, "OBJ"] / vol_base[ajusta]) ** beta
    # D19 · crecimiento de salas por campus × clase que implica el objetivo (con β): sirve para escalar el naive.
    # Solo cuentan los estratos del período base (el naive es su mapa): los que vienen de T−200 o se prestan de un
    # estrato grueso no están en el naive, y los que desaparecen en T suman su volumen base con objetivo 0.
    _u = base[base["EN_BASE"].fillna(False).astype(bool)].drop_duplicates("EST")
    G1_CRECE["ultimo"] = (_u.groupby(["CAMPUS_DESTINO", "CLASE"])["OBJ"].sum()
                          / s1.groupby(["COD_CAMPUS", "CLASE"])["VOL"].sum().rename_axis(["CAMPUS_DESTINO", "CLASE"])).fillna(0.0)
    base["K"] = np.minimum(10, np.ceil(base["OBJ"] / vol_base) + 1).astype(int)

    # Filas de cada sección en arreglos (CSR) para sumar celdas rápido
    filas = filas.sort_values("SEC").reset_index(drop=True)
    f_sec = filas["SEC"].to_numpy()
    f_resto = resto(filas["DIA"].map(DIA_I).to_numpy(), filas["BLOQUE"].to_numpy(), filas["CLASE"].map(CLA_I).to_numpy())
    f_camp, f_peso, f_bloque = filas["CAMP"].map(CAMP).to_numpy(), filas["PESO"].to_numpy(), filas["BLOQUE"].to_numpy()
    rep = base.loc[base.index.repeat(base["K"])].copy()
    rep["COPIA"] = rep.groupby(level=0).cumcount()
    rep = rep.reset_index(drop=True)
    ini = np.searchsorted(f_sec, rep["SEC"].to_numpy(), "left"); fin = np.searchsorted(f_sec, rep["SEC"].to_numpy(), "right")
    r_camp_dest = rep["CAMPUS_DESTINO"].map(CAMP).to_numpy(); r_prest = rep["PRESTADO"].to_numpy()
    r_uni = rep["UNIDAD"].map(UNI).fillna(-1).astype(int).to_numpy()

    if tol is not None:
        # §8 · rango que necesita cada sección: inscritos × razón^(1 − β) ÷ tolerancia de su clase
        _cap = filas[["SEC", "EDIFICIO", "SALA"]].drop_duplicates()
        _cap["CAP"] = _oferta_cap.reindex(pd.MultiIndex.from_frame(_cap[["EDIFICIO", "SALA"]])).to_numpy()
        r_rango = rango_de(tam_necesario(rep["VOL"] * rep["RZ"] ** (1 - (beta if beta is not None else 1.0)), rep["VOL"],
                                         rep["SEC"].map(_cap.groupby("SEC")["CAP"].max()), rep["CLASE"].map(tol).fillna(1.0)))
        s_rango = np.zeros((n_sim, N_CELDAS * NR))
    rng = np.random.default_rng(SEMILLA)
    s_celda, s_uni = np.zeros((n_sim, N_CELDAS)), np.zeros((n_sim, N_UNI))
    elegido_1 = None
    for s in range(n_sim):
        rep["KEY"] = rep["COPIA"] + rng.random(len(rep))
        o = rep.sort_values(["EST", "KEY"])
        acum = o.groupby("EST")["VOL"].cumsum()
        toma = (acum - o["VOL"] / 2) < o["OBJ"]
        idx = o.index[toma.to_numpy()].to_numpy()
        if s == 0:
            elegido_1 = rep.loc[idx]
        largo = fin[idx] - ini[idx]
        filas_i = np.repeat(ini[idx], largo) + (np.arange(largo.sum()) - np.repeat(np.cumsum(largo) - largo, largo))
        camp = np.where(np.repeat(r_prest[idx], largo), np.repeat(r_camp_dest[idx], largo), f_camp[filas_i])
        s_celda[s] = np.bincount(camp * N_SC + f_resto[filas_i], weights=f_peso[filas_i], minlength=N_CELDAS)
        if tol is not None:
            s_rango[s] = np.bincount((camp * N_SC + f_resto[filas_i]) * NR + np.repeat(r_rango[idx], largo), weights=f_peso[filas_i], minlength=N_CELDAS * NR)
        u = np.repeat(r_uni[idx], largo); ok = u >= 0
        s_uni[s] = np.bincount(u[ok] * NB + f_bloque[filas_i][ok], weights=f_peso[filas_i][ok], minlength=N_UNI)
    if tol is not None:
        RANGO_G1["ultimo"] = s_rango
    return s_celda, s_uni, sin_base, elegido_1, filas


RANGO_G1 = {}
G1_CRECE = {}


def vec_crece(crece):
    # D19 · factor de crecimiento por celda (campus × clase); 1 donde no hay dato
    g = np.ones(N_CELDAS)
    d, b = np.repeat(np.arange(len(DIAS)), NB), np.tile(np.arange(NB), len(DIAS))
    for (c, k), v in crece.items():
        if c in CAMP and k in CLA_I and np.isfinite(v):
            g[CAMP[c] * N_SC + resto(d, b, CLA_I[k])] = v
    return g

In [68]:
def asignar_salas(elegidas, filas):
    # Las secciones del año anterior (copia 0, sin préstamo) quedan en su sala; las demás buscan sala libre equivalente
    carga = {}
    cap = oferta.set_index(["EDIFICIO", "SALA"])["CAPACIDAD"].to_dict()
    por_clase = {k: g for k, g in oferta.groupby(["COD_CAMPUS_EDIFICIO", "CLASE_MAPA"])}
    filas_sec = {k: g for k, g in filas.groupby("SEC")}
    deficit, total = 0.0, 0.0
    orden = elegidas.assign(PRIORIDAD=((elegidas["COPIA"] > 0) | elegidas["PRESTADO"]).astype(int)).sort_values("PRIORIDAD")
    for r in orden.itertuples():
        fs = filas_sec.get(r.SEC)
        if fs is None:
            continue
        for (edif, sala, camp, clase), g in fs.groupby(["EDIFICIO", "SALA", "CAMP", "CLASE"]):
            slots = list(zip(g["DIA"], g["BLOQUE"], g["PESO"]))
            total += g["PESO"].sum()
            libre = lambda e, s_: all(carga.get((e, s_, d, b), 0) + p <= 1 + 1e-9 for d, b, p in slots)
            destino = (edif, sala) if (r.PRIORIDAD == 0 and libre(edif, sala)) else None
            if destino is None:
                camp_dest = r.CAMPUS_DESTINO if r.PRESTADO else camp
                cand = por_clase.get((camp_dest, clase))
                if cand is not None:
                    cand = cand[cand["CAPACIDAD"].fillna(0) >= r.INSCRITOS]
                    cand = pd.concat([cand[cand["EDIFICIO"] == edif], cand[cand["EDIFICIO"] != edif]])
                    destino = next(((e, s_) for e, s_ in zip(cand["EDIFICIO"], cand["SALA"]) if libre(e, s_)), None)
            if destino is None:
                deficit += g["PESO"].sum()
                continue
            for d, b, p in slots:
                carga[(destino[0], destino[1], d, b)] = carga.get((destino[0], destino[1], d, b), 0) + p
    return deficit / total if total else 0.0

## 4c · H1 · Programación sintética por hábito de sala (Diego, 26-10)

Si en los últimos años una sala tuvo, por ejemplo, una cátedra TEO los martes de 8:00 a 9:30, lo más probable es que la siga teniendo. Este modelo usa ese **hábito** como motor y deja que la demanda decida **cuántas** secciones hay.

**Decisiones de Diego (26-10):**
- **Ventana del hábito:** todos los años disponibles del mismo sufijo, con más peso para los recientes (año −1: 1; −2: 0,75; −3: 0,5; −4: 0,25).
- **Tipo de sección** = campus × **materia** × componente (más nivel y período). La unidad académica se deriva de la materia, y la materia empalma con `cod_carrera` de Ariadna.
- **Estándar de programación:** el percentil del tamaño de sección por componente y campus que mejor se ajusta (es el método de Euribia). Se calcula sobre el **homólogo del año anterior**, para evitar la fuga H3. El percentil (P25, P50, P75, P90) se elige por componente con los orígenes anteriores; sin orígenes anteriores, P50.

**Pasos:**
1. **Patrones habituales.** Un patrón es la semana de una sección física en una sala: sala × conjunto de (día, bloque). Su **fuerza** es la suma de los pesos de los años en que una sección del mismo tipo lo usó. Las filas del patrón vienen de su aparición más reciente.
2. **Secciones necesarias** por tipo = inscritos ÷ estándar(componente, campus), redondeado y como mínimo 1.
   - **H1:** inscritos reales del objetivo (oráculo de volumen).
   - **H1A:** inscritos del año anterior × crecimiento de la **matrícula de Ariadna** de la carrera (`MATERIA` = `cod_carrera`, sede = campus). Para los departamentos de servicio, el crecimiento de su campus. Es un oráculo de matrícula (Ariadna real), porque no hay pronósticos de Ariadna hechos en el pasado. Los tipos que no existían el año anterior no se proyectan, igual que en una proyección real.
3. **Asignación:** cada tipo toma sus patrones de mayor a menor fuerza hasta cubrir las secciones necesarias. Si faltan patrones, los más habituales se repiten: en la práctica irían a otra sala equivalente del mismo campus y clase, así que la celda del mapa no cambia. Un tipo sin historia usa los patrones de campus × unidad × componente y, si no hay, los de campus × componente.

Es un pronóstico de **punto**; la banda se agrega con H5.

In [69]:
PESO_EDAD = {1: 1.0, 2: 0.75, 3: 0.5, 4: 0.25}
PERCENTILES = [25, 50, 75, 90]
TIPO_H = ["COD_CAMPUS", "MATERIA", "COMPONENTE"]
ariadna = pd.read_parquet(DESCARGAS / "MODELO_PROYECCCIÓN_SIMPLE" / "PROYECCION_MATRICULA_2026_2031_28092026.parquet",
                          columns=["periodo", "cod_carrera", "sede", "n", "tipo"])
ariadna = ariadna[ariadna["tipo"] == "REAL"].assign(periodo=lambda d: d["periodo"].astype(int))
mat_ari = ariadna.groupby(["periodo", "cod_carrera", "sede"])["n"].sum()
camp_ari = ariadna.groupby(["periodo", "sede"])["n"].sum()
_cache_sf = {}


def sf(P, T):
    # Secciones físicas del período P, clasificadas solo con información anterior a T
    if (P, T) not in _cache_sf:
        _cache_sf[(P, T)] = secciones_fisicas(P, clasificar([p for p in pl["PERIODO"].unique() if p < T]))
    return _cache_sf[(P, T)]


def estandar(T, k):
    # Percentil k del tamaño de sección por componente × campus en el homólogo anterior (T−100); respaldo: componente
    s, _ = sf(T - 100, T)
    s = s[s["INSCRITOS"] > 0]
    return (s.groupby(["COMPONENTE", "COD_CAMPUS"])["INSCRITOS"].quantile(k / 100),
            s.groupby("COMPONENTE")["INSCRITOS"].quantile(k / 100))


def n_secciones(vol, T, k_por_comp):
    # vol: TIPO_H + INSCRITOS → secciones necesarias por tipo
    out = vol.copy()
    est = pd.Series(np.nan, index=out.index)
    for comp, g in out.groupby("COMPONENTE"):
        fino, grueso = estandar(T, k_por_comp.get(comp, 50))
        e = pd.Series([fino.get((comp, c), np.nan) for c in g["COD_CAMPUS"]], index=g.index)
        est.loc[g.index] = e.fillna(grueso.get(comp, np.nan))
    out["ESTANDAR"] = est
    out["N"] = np.maximum(1, np.round(out["INSCRITOS"] / out["ESTANDAR"].clip(lower=1))).fillna(1).astype(int)
    return out


def elegir_percentil(T):
    # Por componente, el percentil cuyo número de secciones mejor calza con lo real en los orígenes anteriores del mismo nivel
    previos = [t for t in ORIGENES if t < T and TIPO_POR_SUF[t % 100] == TIPO_POR_SUF[T % 100]]
    if not previos:
        return {}
    err = {}
    for t in previos:
        s, _ = sf(t, t + 1)
        real = s.groupby(TIPO_H).agg(INSCRITOS=("INSCRITOS", "sum"), N_REAL=("SEC", "size")).reset_index()
        for k in PERCENTILES:
            p = n_secciones(real[TIPO_H + ["INSCRITOS"]], t, {c: k for c in real["COMPONENTE"].unique()})
            p["N_REAL"] = real["N_REAL"]
            for comp, g in p.groupby("COMPONENTE"):
                acc = err.setdefault(comp, {}).setdefault(k, [0.0, 0.0])
                acc[0] += (g["N"] - g["N_REAL"]).abs().sum(); acc[1] += g["N_REAL"].sum()
    return {c: min(e, key=lambda k: e[k][0] / max(e[k][1], 1)) for c, e in err.items()}


def patrones(T):
    # Patrones habituales por tipo con su fuerza, desde los años anteriores del mismo sufijo
    filas_pat = []
    for edad, w in PESO_EDAD.items():
        P = T - 100 * edad
        if P not in set(pl["PERIODO"].unique()):
            continue
        s, f = sf(P, T)
        f = f.merge(s[["SEC", *TIPO_H, "UNIDAD"]], on="SEC")
        f = f.sort_values(["SEC", "DIA", "BLOQUE"])
        f["_DB"] = f["DIA"].str[:2] + f["BLOQUE"].astype(str)
        clave = (f.groupby("SEC").agg(E=("EDIFICIO", "first"), S=("SALA", "first"), DB=("_DB", ",".join))
                  .assign(PATRON=lambda d: d["E"] + "|" + d["S"] + "|" + d["DB"])["PATRON"])
        filas_pat.append(f.join(clave, on="SEC").assign(PESO_AÑO=w, EDAD=edad))
    f = pd.concat(filas_pat, ignore_index=True)
    fuerza = (f.drop_duplicates(["SEC", "EDAD"]).groupby(TIPO_H + ["UNIDAD", "PATRON"])
               .agg(FUERZA=("PESO_AÑO", "sum"), EDAD_MIN=("EDAD", "min")).reset_index()
               .sort_values(["FUERZA", "EDAD_MIN"], ascending=[False, True]))
    rep = f.sort_values("EDAD").drop_duplicates(TIPO_H + ["PATRON"])[TIPO_H + ["PATRON", "SEC", "EDAD"]]
    return fuerza, f.merge(rep, on=TIPO_H + ["PATRON", "SEC", "EDAD"])


def generar_h1(T, modo):
    k = elegir_percentil(T)
    fuerza, filas_rep = patrones(T)
    if modo == "H1":
        s, _ = sf(T, T + 1)
        vol = s.groupby(TIPO_H)["INSCRITOS"].sum().reset_index()
    else:
        s, _ = sf(T - 100, T)
        vol = s.groupby(TIPO_H)["INSCRITOS"].sum().reset_index()
        idx_m = lambda P: pd.MultiIndex.from_arrays([np.full(len(vol), P), vol["MATERIA"], vol["COD_CAMPUS"]])
        idx_c = lambda P: pd.MultiIndex.from_arrays([np.full(len(vol), P), vol["COD_CAMPUS"]])
        g_mat = mat_ari.reindex(idx_m(T)).to_numpy() / mat_ari.reindex(idx_m(T - 100)).to_numpy()
        g_camp = camp_ari.reindex(idx_c(T)).to_numpy() / camp_ari.reindex(idx_c(T - 100)).to_numpy()
        crec = np.where(np.isfinite(g_mat) & (g_mat > 0), g_mat, np.where(np.isfinite(g_camp) & (g_camp > 0), g_camp, 1.0))
        vol["INSCRITOS"] = vol["INSCRITOS"] * crec
    nec = n_secciones(vol, T, k)
    nec["UNIDAD"] = nec["MATERIA"].map(UNIDAD_MAT).fillna("SIN UNIDAD")

    pat_tipo = {k_: g for k_, g in fuerza.groupby(TIPO_H)}
    pat_uni = {k_: g for k_, g in fuerza.groupby(["COD_CAMPUS", "UNIDAD", "COMPONENTE"])}
    pat_cc = {k_: g for k_, g in fuerza.groupby(["COD_CAMPUS", "COMPONENTE"])}
    elegidos = []
    for r in nec.itertuples(index=False):
        g = pat_tipo.get((r.COD_CAMPUS, r.MATERIA, r.COMPONENTE))
        if g is None:
            g = pat_uni.get((r.COD_CAMPUS, r.UNIDAD, r.COMPONENTE), pat_cc.get((r.COD_CAMPUS, r.COMPONENTE)))
        if g is None or not len(g):
            continue
        e = g.iloc[np.arange(r.N) % len(g)][TIPO_H + ["PATRON"]].copy()
        e["UNIDAD_DESTINO"] = r.UNIDAD
        elegidos.append(e)
    filas = pd.concat(elegidos, ignore_index=True).merge(filas_rep, on=TIPO_H + ["PATRON"], how="inner")
    celda = np.bincount(_celda(filas.rename(columns={"CAMP": "COD_CAMPUS_EDIFICIO"})), weights=filas["PESO"].to_numpy(), minlength=N_CELDAS)
    u = filas["UNIDAD_DESTINO"].map(UNI).fillna(-1).astype(int).to_numpy(); ok = u >= 0
    uni = np.bincount(u[ok] * NB + filas["BLOQUE"].to_numpy()[ok], weights=filas["PESO"].to_numpy()[ok], minlength=N_UNI)
    return celda, uni, {"SECCIONES": int(nec["N"].sum()), "PERCENTIL_POR_COMPONENTE": str(k)}

## 4d · H2 · Hábito de sala con el estándar real de programación (Diego, 26-10)

Corrige el punto débil de H1, que era contar las secciones:
- **Estándar real** por sede × componente: el percentil del tamaño de sección física (P5…P95) cuyo número implícito de secciones (inscritos ÷ Pk) reproduce las secciones que de verdad se programaron en el homólogo anterior (T−100). Es un dato real del pasado, no el estándar teórico (TEO = 80, LAB = 30…).
- **Secciones** por sede × componente = inscritos ÷ estándar real. Se usan los inscritos reales del objetivo, el mismo oráculo de volumen que G1b, para comparar en igualdad.
- **Reparto** entre materias según su peso en los inscritos (resto mayor, sin mínimo forzado).
- **Ubicación** por hábito de sala, como H1.

**C2** = w · N0 + (1 − w) · H2, con w calibrado igual que C1: solo con los orígenes anteriores del mismo nivel.

In [70]:
KS_REAL = list(range(5, 100, 5))
SC_H = ["COD_CAMPUS", "COMPONENTE"]


def estandar_real(s):
    s = s[s["INSCRITOS"] > 0]
    out = {}
    for key, g in s.groupby(SC_H):
        ins, n = g["INSCRITOS"].sum(), len(g)
        q = {k: np.percentile(g["INSCRITOS"], k) for k in KS_REAL}
        out[key] = q[min(KS_REAL, key=lambda k: abs(ins / max(q[k], 1) - n))]
    return pd.Series(out)


def repartir_por_materia(n_sc, ins_tipo):
    filas = []
    for key, g in ins_tipo.groupby(SC_H):
        n = int(n_sc[key]) if key in n_sc.index and pd.notna(n_sc[key]) else len(g)
        cuota = (g["INSCRITOS"] / g["INSCRITOS"].sum() * n).to_numpy()
        base = np.floor(cuota).astype(int)
        falta = n - base.sum()
        if falta > 0:
            base[np.argsort(-(cuota - base))[:falta]] += 1
        filas.append(g.assign(N=base))
    return pd.concat(filas, ignore_index=True)


def generar_h2(T):
    sT, _ = sf(T, T + 1)
    sA, _ = sf(T - 100, T)
    est = estandar_real(sA)
    ins_sc = sT.groupby(SC_H)["INSCRITOS"].sum()
    n_sc = np.round(ins_sc / est.reindex(ins_sc.index).clip(lower=1))
    n_sc = n_sc.fillna(sA.groupby(SC_H).size().reindex(ins_sc.index)).fillna(1)
    reparto = repartir_por_materia(n_sc, sT.groupby(TIPO_H)["INSCRITOS"].sum().reset_index())
    reparto["UNIDAD"] = reparto["MATERIA"].map(UNIDAD_MAT).fillna("SIN UNIDAD")
    fuerza, filas_rep = patrones(T)
    pat_tipo = {k_: g for k_, g in fuerza.groupby(TIPO_H)}
    pat_sc = {k_: g for k_, g in fuerza.groupby(SC_H)}
    elegidos = []
    for r in reparto[reparto["N"] > 0].itertuples(index=False):
        g = pat_tipo.get((r.COD_CAMPUS, r.MATERIA, r.COMPONENTE), pat_sc.get((r.COD_CAMPUS, r.COMPONENTE)))
        if g is None or not len(g):
            continue
        e = g.iloc[np.arange(r.N) % len(g)][TIPO_H + ["PATRON"]].copy()
        e["UNIDAD_DESTINO"] = r.UNIDAD
        elegidos.append(e)
    filas = pd.concat(elegidos, ignore_index=True).merge(filas_rep, on=TIPO_H + ["PATRON"], how="inner")
    celda = np.bincount(_celda(filas.rename(columns={"CAMP": "COD_CAMPUS_EDIFICIO"})), weights=filas["PESO"].to_numpy(), minlength=N_CELDAS)
    u = filas["UNIDAD_DESTINO"].map(UNI).fillna(-1).astype(int).to_numpy(); ok = u >= 0
    uni = np.bincount(u[ok] * NB + filas["BLOQUE"].to_numpy()[ok], weights=filas["PESO"].to_numpy()[ok], minlength=N_UNI)
    n_real = sT.groupby(SC_H).size().reindex(n_sc.index)
    return celda, uni, {"SECCIONES": int(n_sc.sum()), "ERROR_N_SEDE_COMP": float((n_sc - n_real).abs().sum() / n_real.sum()),
                        "SESGO_N": float(n_sc.sum() / n_real.sum() - 1)}

## 4e · Volumen proyectado en vez del oráculo (26-10)

Hasta aquí, G1b y C1 crecían hasta los inscritos **reales** del objetivo, que funcionan como oráculo. Ahora G1b crece hasta un volumen **proyectado**: las secciones físicas de T−100 escaladas por una razón de crecimiento por tipo (campus × materia × componente), con la misma regla que `Modelos.ipynb` §3. Hay tres fuentes de razón:

| Variante | Razón por tipo | Qué prueba |
|---|---|---|
| **EUR** · Euribia | Σ P50 ÷ Σ inscritos del año anterior, dentro de Euribia | si el modelo de demanda aporta |
| **PER** · persistencia | 1 (el volumen del año anterior) | el rival «nada cambia» |
| **TEN** · tendencia | crecimiento T−100 ÷ T−200 de los inscritos del campus, mismo sufijo | el rival ingenuo de Euribia |

**Fuente de Euribia:**
- **202410–202615:** filas `REAL` de `SERIE_COMPLETA`. Cada año se reentrenó solo con años anteriores. La razón se calcula sobre las asignaturas dictadas en T **y** en T−100, contra el real de T−100 de la misma serie.
- **202620 y 202625:** el entregable. P50 ÷ `ins_lag_1y`, con `ins_lag_1y` > 0.
- Los tipos sin razón quedan en 1 y OCS queda fuera.

**Dos ventajas de Euribia en la historia** (el resultado de EUR es una cota optimista antes de 2026):
- La serie usa la grilla que se dictó, y la intersección con T−100 solo la neutraliza en parte.
- Su nivel agregado se fija con `TASA_NIVEL` (0,9878), la media de 11 años cerrados que incluye el año objetivo.

Cada variante se combina con el naive en **C1-EUR / C1-PER / C1-TEN**, con w recalibrado por variante solo con los orígenes anteriores del mismo nivel.

`VALIDACION_VOLUMEN.parquet` mide además el volumen por sí solo:
- WAPE de los inscritos proyectados por tipo y por estrato contra el real, y su sesgo;
- en Euribia, la cobertura P10–P90 por tipo;
- el error del número de secciones del estándar real (H2) partiendo de cada volumen.

In [71]:
TIPO_V = ["COD_CAMPUS", "MATERIA", "COMPONENTE"]
CLAVE_EUR = ["MATERIA", "CURSO", "COMPONENTE", "CAMPUS"]
COL_PROY = {"PROY_BAJO": "P10", "PROY_ESTANDAR": "P50", "PROY_ALTO": "P90"}


def razon_euribia(T, col="PROY_ESTANDAR"):
    if T in PERIODOS_PRONOSTICO_EURIBIA:
        e = eur_proy[eur_proy["PERIODO"] == T]
        e = e.assign(NUM=e[COL_PROY[col]], DEN=e["ins_lag_1y"])
    else:
        r = eur_serie[eur_serie["TIPO"] == "REAL"]
        a = r[r["PERIODO"] == T].groupby(CLAVE_EUR)[[col]].sum()
        b = r[r["PERIODO"] == T - 100].groupby(CLAVE_EUR)[["INSCRITOS_REAL"]].sum()
        e = a.join(b, how="inner").reset_index().rename(columns={col: "NUM", "INSCRITOS_REAL": "DEN"})
    e = e[~e["COMPONENTE"].isin(COMPONENTES_SIN_HORARIO) & (e["DEN"] > 0)]
    g = e.groupby(["CAMPUS", "MATERIA", "COMPONENTE"])[["NUM", "DEN"]].sum()
    g.index.names = TIPO_V
    return (g["NUM"] / g["DEN"]).clip(0.25, 4.0).rename("RAZON")


def razon_tendencia(T):
    ins = ph[ph["PERIODO"].isin([T - 100, T - 200])].groupby(["PERIODO", "COD_CAMPUS"])["N_INSCRITOS"].sum().unstack(0)
    return (ins[T - 100] / ins[T - 200]).replace([np.inf, -np.inf], np.nan).dropna().clip(0.25, 4.0).rename("RAZON")


def razones(T):
    return {"EUR": razon_euribia(T), "PER": pd.Series(1.0, index=pd.Index(sorted(CAMP), name="COD_CAMPUS"), name="RAZON"),
            "TEN": razon_tendencia(T)}


def aplicar_razon(s, razon):
    claves = list(razon.index.names)
    return s.merge(razon.reset_index(), on=claves, how="left").fillna({"RAZON": 1.0})


def metricas_volumen(T, nombre, razon):
    sA, _ = sf(T - 100, T)
    sT, _ = sf(T, T + 1)
    p = aplicar_razon(sA, razon).assign(PROY=lambda d: d["INSCRITOS"] * d["RAZON"])
    out = {"ORIGEN": T, "USO": ORIGENES[T], "VARIANTE": nombre}
    for grano, claves in (("tipo", TIPO_V), ("estrato", ESTRATO)):
        j = pd.concat([p.groupby(claves)["PROY"].sum(), sT.groupby(claves)["INSCRITOS"].sum().rename("REAL")], axis=1).fillna(0)
        out[f"WAPE_{grano.upper()}"] = float((j["PROY"] - j["REAL"]).abs().sum() / j["REAL"].sum())
    out["SESGO_TOTAL"] = float(p["PROY"].sum() / sT["INSCRITOS"].sum() - 1)
    # Número de secciones con el estándar real (H2), partiendo de este volumen
    est = estandar_real(sA)
    n_sc = np.round(p.groupby(SC_H)["PROY"].sum() / est.reindex(p.groupby(SC_H).size().index).clip(lower=1))
    n_real = sT.groupby(SC_H).size()
    j = pd.concat([n_sc.rename("N"), n_real.rename("R")], axis=1).fillna(0)
    out["ERROR_N_SEDE_COMP"] = float((j["N"] - j["R"]).abs().sum() / j["R"].sum())
    out["SESGO_N"] = float(j["N"].sum() / j["R"].sum() - 1)
    if nombre == "EUR":
        lo, hi = (aplicar_razon(sA, razon_euribia(T, c)).assign(V=lambda d: d["INSCRITOS"] * d["RAZON"]).groupby(TIPO_V)["V"].sum()
                  for c in ("PROY_BAJO", "PROY_ALTO"))
        rt = sT.groupby(TIPO_V)["INSCRITOS"].sum()
        j = pd.concat([lo.rename("LO"), hi.rename("HI"), rt.rename("R")], axis=1).dropna()
        out["COBERTURA_P10_P90_TIPO"] = float(j.loc[(j["R"] >= j["LO"]) & (j["R"] <= j["HI"]), "R"].sum() / j["R"].sum())
        out["COBERTURA_RAZON_EUR"] = float(aplicar_razon(sA, razon).pipe(
            lambda d: d.loc[d.set_index(TIPO_V).index.isin(razon.index), "INSCRITOS"].sum() / d["INSCRITOS"].sum()))
    return out

## 5 · Corrida

In [72]:
filas, filas_vol = [], []
cache_real = {}
PRED = {}
def real(T):
    if T not in cache_real:
        cache_real[T] = mapa_real(T)
    return cache_real[T]

for T, uso in ORIGENES.items():
    r_celda, r_uni = real(T)
    granos_r = {"celda": r_celda, "campus × día × bloque": agr_cdb(r_celda), "unidad × bloque": r_uni}
    modelos = {}
    n1c, n1u = real(T - 100); n2c, n2u = real(T - 200)
    modelos["N0"] = ({"celda": n1c, "campus × día × bloque": agr_cdb(n1c), "unidad × bloque": n1u}, {})
    nc, nu = 2 / 3 * n1c + 1 / 3 * n2c, 2 / 3 * n1u + 1 / 3 * n2u
    modelos["N1"] = ({"celda": nc, "campus × día × bloque": agr_cdb(nc), "unidad × bloque": nu}, {})
    sc, su, sh = simular(atrib[atrib["PERIODO"] == T][["COD_SERIE", "COD_MATERIA", "COD_COMPONENTE", "COD_CAMPUS"]], T)
    modelos["P1"] = ({"celda": sc, "campus × día × bloque": agr_cdb(sc), "unidad × bloque": su}, {"SIN_HORARIO": sh})
    gc, gu, sin_base, eleg, fil = generar_g1(T)
    modelos["G1"] = ({"celda": gc, "campus × día × bloque": agr_cdb(gc), "unidad × bloque": gu},
                     {"SIN_HORARIO": sin_base, "DEFICIT_SALAS": asignar_salas(eleg, fil)})
    beta, pares = elasticidad(T)
    gc, gu, sin_base, eleg, fil = generar_g1(T, beta=beta)
    crece = {"G1b": vec_crece(G1_CRECE["ultimo"])}
    modelos["G1b"] = ({"celda": gc, "campus × día × bloque": agr_cdb(gc), "unidad × bloque": gu},
                      {"SIN_HORARIO": sin_base, "DEFICIT_SALAS": asignar_salas(eleg, fil), "BETA": beta,
                       "PARES_CALIBRACION": ",".join(map(str, pares))})
    for nombre_v, rz in razones(T).items():
        gc, gu, *_ = generar_g1(T, beta=beta, razon=rz)
        crece[f"G1b-{nombre_v}"] = vec_crece(G1_CRECE["ultimo"])
        modelos[f"G1b-{nombre_v}"] = ({"celda": gc, "campus × día × bloque": agr_cdb(gc), "unidad × bloque": gu}, {"BETA": beta})
        filas_vol.append(metricas_volumen(T, nombre_v, rz))
    for nombre in ("H1", "H1A"):
        hc, hu, extra_h = generar_h1(T, nombre)
        modelos[nombre] = ({"celda": hc, "campus × día × bloque": agr_cdb(hc), "unidad × bloque": hu}, extra_h)
    hc, hu, extra_h = generar_h2(T)
    modelos["H2"] = ({"celda": hc, "campus × día × bloque": agr_cdb(hc), "unidad × bloque": hu}, extra_h)
    for nombre, var in (("E1", "D2D8"), ("E1s", "TEO")):
        d, grilla, n_secc, explicado = secciones(T, var)
        sc, su, sh = simular(d, T)
        modelos[nombre] = ({"celda": sc, "campus × día × bloque": agr_cdb(sc), "unidad × bloque": su},
                           {"GRILLA": grilla, "SECCIONES": n_secc, "NRC_REALES": int((atrib["PERIODO"] == T).sum()),
                            "SIN_HORARIO": sh, "VERDAD_EXPLICADA_POR_PROYECCION": explicado})
    for mod, (pred, extra) in modelos.items():
        for grano, rv in granos_r.items():
            met = metricas(rv, pred[grano])
            filas.append({"ORIGEN": T, "USO": uso, "MODELO": mod, "GRANO": grano, **met, **extra})
    PRED[T] = {"real": granos_r, "N0": modelos["N0"][0], "G1b": modelos["G1b"][0], "H2": modelos["H2"][0], "CRECE": crece,
               **{k: modelos[k][0] for k in ("G1b-EUR", "G1b-PER", "G1b-TEN")}}
    print(f"{T} listo")

# C1 · combinación w·N0 + (1−w)·G1b. w se elige solo con los orígenes ANTERIORES del mismo nivel (UG o AV),
# minimizando el WAPE medio en los dos granos para decidir. Sin orígenes anteriores, w = 1 (= naive).
GRILLA_W = np.round(np.arange(0, 1.01, 0.1), 1)
COMB = {}
GRANOS_DECIDEN = ["celda", "campus × día × bloque"]
nivel_de = lambda t: "UG" if t % 100 in (10, 20) else "AV"
# D19 · C1E = C1 con el naive escalado por el crecimiento de salas que implica G1b en cada campus × clase
def naive_de(t, g, comp_c, escala):
    if not escala or g == "unidad × bloque":
        return PRED[t]["N0"][g]
    v = PRED[t]["N0"]["celda"] * PRED[t]["CRECE"][comp_c]
    return v if g == "celda" else agr_cdb(v)


COMBINACIONES = [("C1", "G1b"), ("C2", "H2"), ("C1-EUR", "G1b-EUR"), ("C1-PER", "G1b-PER"), ("C1-TEN", "G1b-TEN"),
                 ("C1E", "G1b"), ("C1E-EUR", "G1b-EUR"), ("C1E-PER", "G1b-PER"), ("C1E-TEN", "G1b-TEN")]
for nombre_c, comp_c in COMBINACIONES:
    escala = nombre_c.startswith("C1E")
    for T, uso in ORIGENES.items():
        previos = [t for t in PRED if t < T and nivel_de(t) == nivel_de(T)]
        media = lambda t, g: PRED[t][comp_c][g].mean(0) if PRED[t][comp_c][g].ndim == 2 else PRED[t][comp_c][g]
        if previos:
            err = {w: np.mean([wape(PRED[t]["real"][g], w * naive_de(t, g, comp_c, escala) + (1 - w) * media(t, g))
                               for t in previos for g in GRANOS_DECIDEN]) for w in GRILLA_W}
            w = min(err, key=err.get)
        else:
            w = 1.0
        for grano, rv in PRED[T]["real"].items():
            comp_pred = PRED[T][comp_c][grano]
            nv = naive_de(T, grano, comp_c, escala)
            comb = w * nv[None, :] + (1 - w) * comp_pred if comp_pred.ndim == 2 else w * nv + (1 - w) * comp_pred
            COMB[(nombre_c, T, grano)] = comb
            filas.append({"ORIGEN": T, "USO": uso, "MODELO": nombre_c, "GRANO": grano, **metricas(rv, comb),
                          "W_NAIVE": w, "ORIGENES_CALIBRACION": ",".join(map(str, previos))})

res = pd.DataFrame(filas)
base_n0 = res[res["MODELO"] == "N0"].set_index(["ORIGEN", "GRANO"])["WAPE"]
res["MASE"] = res["WAPE"] / res.set_index(["ORIGEN", "GRANO"]).index.map(base_n0)
res.to_parquet(BBDD / "VALIDACION_RESULTADOS.parquet", index=False)
vol_val = pd.DataFrame(filas_vol)
vol_val.to_parquet(BBDD / "VALIDACION_VOLUMEN.parquet", index=False)

C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:70: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  _u = base[base["EN_BASE"].fillna(False).astype(bool)].drop_duplicates("EST")
C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:66: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[5.84402669e+02 1.60588455e+03 4.62123919e+03 ... 2.74060909e+00
 2.74060909e+00 1.53739122e+00]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  base.loc[ajusta, "OBJ"] = vol_base[ajusta] * (base.loc[ajusta, "OBJ"] / vol_base[ajusta]) ** beta
C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:70: FutureWarning: Downcasting object

202410 listo


C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:70: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  _u = base[base["EN_BASE"].fillna(False).astype(bool)].drop_duplicates("EST")
C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:66: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[1446.51675484 2563.16432847  340.15191219 ...   18.90482846    4.55249779
    4.55249779]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  base.loc[ajusta, "OBJ"] = vol_base[ajusta] * (base.loc[ajusta, "OBJ"] / vol_base[ajusta]) ** beta
C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:70: FutureWarning: Downcasting object dtype

202420 listo


C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:70: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  _u = base[base["EN_BASE"].fillna(False).astype(bool)].drop_duplicates("EST")
C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:66: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[191.63190611  80.69955675 446.83024269 ...  44.58585513  34.93866501
  53.86061373]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  base.loc[ajusta, "OBJ"] = vol_base[ajusta] * (base.loc[ajusta, "OBJ"] / vol_base[ajusta]) ** beta
C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:70: FutureWarning: Downcasting object dtype array

202520 listo


... (10 outputs left out)

C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:70: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  _u = base[base["EN_BASE"].fillna(False).astype(bool)].drop_duplicates("EST")
C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:66: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[626.94278141 894.4038964  672.87653518 ...  43.39383599  12.83234368
  29.47669974]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  base.loc[ajusta, "OBJ"] = vol_base[ajusta] * (base.loc[ajusta, "OBJ"] / vol_base[ajusta]) ** beta
C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:70: FutureWarning: Downcasting object dtype array

202615 listo


C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:70: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  _u = base[base["EN_BASE"].fillna(False).astype(bool)].drop_duplicates("EST")
C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:66: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[422.69548555 737.6354557  422.69548555 ...  35.40536599  35.40536599
  35.40536599]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  base.loc[ajusta, "OBJ"] = vol_base[ajusta] * (base.loc[ajusta, "OBJ"] / vol_base[ajusta]) ** beta
C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:70: FutureWarning: Downcasting object dtype array

202625 listo


**D19 · naive escalado (27-10).** A largo plazo el naive es una copia del período base que no crece. Con w = 0,8, el centro del mapa crecía solo un 4 % entre 2027 y 2031 mientras los inscritos crecían un 24 %, y en §7 C1 quedaba un 7,6 % bajo lo real en H+4. **C1E** escala el naive de cada campus × clase por el crecimiento de salas que implica el objetivo de G1b (Σ volumen objetivo con β ÷ Σ volumen base) y recalibra w igual que C1. Se adopta si corrige el sesgo sin empeorar el MASE (Diego, 27-10).

## 5b · H5 · Calibración conformal de la banda P10–P90 (26-10)

La banda de las simulaciones solo mide el ruido del remuestreo de G1b, y C1 lo achica a (1 − w): cubre el real en el 1–12 % de las celdas. Se reemplaza por una banda **conformal** construida con los errores pasados del propio modelo (Tomo 10).

**Paso a paso:**
- **Residuo normalizado** de cada celda activa: e = (real − punto) ÷ √(punto + 1), donde punto es la media de C.
- **Estratos:** nivel × clase × tramo del punto (0–5, 5–20, 20–50 y más de 50 salas). En campus × día × bloque se suman las clases, así que el estrato es nivel × tramo.
- **Banda del origen T:** cuantiles de e en los orígenes **anteriores** a T del mismo nivel, nunca en T. Si un estrato tiene menos de 30 residuos, se usa el nivel completo.
- **Orígenes sin anteriores** (el primero de cada nivel): residuos del **naive** en los pares de años previos (P contra P − 100, con P = T − 100 y T − 200), que salen del histórico sin entrenar nada.
- **P10 / P90** = punto + q10 / q90 × √(punto + 1), con piso en 0. Cada grano se calibra aparte.

**Métricas:**
- cobertura P10–P90 (objetivo 80 %, aceptable entre 75 % y 85 %);
- ancho medio;
- puntuación de intervalo de Winkler relativa (con α = 0,2: castiga el ancho y, con factor 10, cada sala que el real queda fuera).

**Regla de adopción:** cobertura dentro de [75 %, 85 %] en la mayoría de los orígenes de pregrado que deciden, en celda y en campus × día × bloque.

**Salidas para `Modelos.ipynb`:**
- `VALIDACION_BANDA.parquet`: la banda sin calibrar contra la conformal, por origen.
- `BANDA_CUANTILES.parquet`: q05…q95 por modelo × nivel × grano × clase × tramo, con **todos** los orígenes. Es la calibración vigente para proyectar.

In [73]:
TRAMOS = [0, 5, 20, 50, np.inf]
QS = np.round(np.arange(0.05, 0.96, 0.05), 2)
MIN_RESIDUOS = 30
MODELOS_BANDA = ["C1", "C1-EUR", "C1-PER", "C1-TEN", "C1E", "C1E-EUR", "C1E-PER", "C1E-TEN"]


def clase_de(grano, n):
    return np.array(CLASES)[np.arange(n) % len(CLASES)] if grano == "celda" else np.full(n, "TODAS")


def residuos(real_v, punto, grano):
    act = (real_v > 0) | (punto > 0)
    return pd.DataFrame({"E": (real_v - punto)[act] / np.sqrt(punto[act] + 1), "CLASE": clase_de(grano, len(punto))[act],
                         "TRAMO": pd.cut(punto[act], TRAMOS, right=False, labels=False)})


def cuantiles(res_df):
    # q por clase × tramo; los estratos con pocos residuos heredan los del nivel completo
    glob = res_df["E"].quantile(QS).to_numpy()
    out = {}
    for (c, t), g in res_df.groupby(["CLASE", "TRAMO"]):
        out[(c, t)] = g["E"].quantile(QS).to_numpy() if len(g) >= MIN_RESIDUOS else glob
    return out, glob


def banda(punto, grano, q, glob, niveles=(0.10, 0.90)):
    cl = clase_de(grano, len(punto)); tr = pd.cut(punto, TRAMOS, right=False, labels=False)
    idx = [int(np.argmin(np.abs(QS - a))) for a in niveles]
    Q = np.stack([q.get((c, t), glob) for c, t in zip(cl, tr)])
    esc = np.sqrt(punto + 1)
    return [np.clip(punto + Q[:, i] * esc, 0, None) for i in idx]


def interval_score(y, lo, hi, alfa=0.2):
    return (hi - lo) + 2 / alfa * np.clip(lo - y, 0, None) + 2 / alfa * np.clip(y - hi, 0, None)


def residuos_naive(T, grano):
    out = []
    for P in (T - 100, T - 200):
        if (P - 100) in set(pl["PERIODO"].unique()):
            (rc, _), (nc, _) = real(P), real(P - 100)
            rv, pv = (rc, nc) if grano == "celda" else (agr_cdb(rc), agr_cdb(nc))
            out.append(residuos(rv, pv, grano))
    return pd.concat(out, ignore_index=True) if out else None


filas_banda, RES = [], {}
for m in MODELOS_BANDA:
    for T in ORIGENES:
        for g in GRANOS_DECIDEN:
            c = COMB[(m, T, g)]
            RES[(m, T, g)] = residuos(PRED[T]["real"][g], c.mean(0) if c.ndim == 2 else c, g)
for m in MODELOS_BANDA:
    for T, uso in ORIGENES.items():
        previos = [t for t in ORIGENES if t < T and nivel_de(t) == nivel_de(T)]
        for g in GRANOS_DECIDEN:
            rv, c = PRED[T]["real"][g], COMB[(m, T, g)]
            punto = c.mean(0) if c.ndim == 2 else c
            act = (rv > 0) | (punto > 0)
            if c.ndim == 2:
                lo_s, hi_s = np.percentile(c, 10, 0), np.percentile(c, 90, 0)
            else:
                lo_s = hi_s = punto
            cal = pd.concat([RES[(m, t, g)] for t in previos], ignore_index=True) if previos else residuos_naive(T, g)
            q, glob = cuantiles(cal)
            lo_c, hi_c = banda(punto, g, q, glob)
            for metodo, lo, hi in (("sin calibrar", lo_s, hi_s), ("conformal", lo_c, hi_c)):
                y, l, h = rv[act], lo[act], hi[act]
                filas_banda.append({"ORIGEN": T, "USO": uso, "MODELO": m, "GRANO": g, "METODO": metodo,
                                    "COBERTURA": float(((y >= l) & (y <= h)).mean()), "ANCHO_MEDIO": float((h - l).mean()),
                                    "IS_REL": float(interval_score(y, l, h).sum() / y.sum()),
                                    "N_CALIBRACION": len(cal), "FUENTE_CALIBRACION": ",".join(map(str, previos)) or "naive T-100/T-200"})
banda_val = pd.DataFrame(filas_banda)
banda_val.to_parquet(BBDD / "VALIDACION_BANDA.parquet", index=False)

# Calibración vigente para proyectar: todos los orígenes de cada nivel
filas_q = []
for m in MODELOS_BANDA:
    for niv in ("UG", "AV"):
        for g in GRANOS_DECIDEN:
            cal = pd.concat([RES[(m, t, g)] for t in ORIGENES if nivel_de(t) == niv], ignore_index=True)
            q, glob = cuantiles(cal)
            n = cal.groupby(["CLASE", "TRAMO"]).size()
            for c in ["TODAS"] if g != "celda" else CLASES:
                for t in range(len(TRAMOS) - 1):
                    filas_q.append({"MODELO": m, "NIVEL": niv, "GRANO": g, "CLASE": c, "TRAMO": t,
                                    "N_RESIDUOS": int(n.get((c, t), 0)), **{f"Q{int(round(a * 100)):02d}": v for a, v in zip(QS, q.get((c, t), glob))}})
pd.DataFrame(filas_q).to_parquet(BBDD / "BANDA_CUANTILES.parquet", index=False)
cov = banda_val.pivot_table(index=["MODELO", "GRANO", "METODO"], columns="ORIGEN", values="COBERTURA")
cov.round(3)

ORIGEN                                      202410  202415  202420  202425  202520  202525  202610  202615  202620  202625
MODELO  GRANO                 METODO                                                                                      
C1      campus × día × bloque conformal      0.791   0.804   0.697   0.638   0.671   0.723   0.819   0.704   0.874   0.685
                              sin calibrar   0.011   0.010   0.057   0.181   0.055   0.215   0.053   0.224   0.102   0.198
        celda                 conformal      0.799   0.790   0.737   0.603   0.694   0.797   0.790   0.757   0.820   0.734
                              sin calibrar   0.027   0.052   0.067   0.172   0.062   0.234   0.067   0.227   0.108   0.140
C1-EUR  campus × día × bloque conformal      0.791   0.804   0.705   0.624   0.643   0.700   0.816   0.728   0.862   0.668
                              sin calibrar   0.011   0.010   0.072   0.007   0.029   0.081   0.000   0.140   0.000   0.121
        celda                 conformal      0.799   0.790   0.732   0.560   0.667   0.752   0.787   0.777   0.808   0.715
                              sin calibrar   0.027   0.052   0.072   0.031   0.062   0.083   0.009   0.178   0.015   0.119
C1-PER  campus × día × bloque conformal      0.791   0.804   0.707   0.624   0.647   0.692   0.816   0.736   0.862   0.647
                              sin calibrar   0.011   0.010   0.022   0.007   0.010   0.035   0.000   0.012   0.000   0.017
        celda                 conformal      0.799   0.790   0.725   0.560   0.675   0.748   0.797   0.786   0.808   0.698
                              sin calibrar   0.027   0.052   0.015   0.031   0.011   0.049   0.009   0.041   0.015   0.024
C1-TEN  campus × día × bloque conformal      0.791   0.804   0.685   0.624   0.598   0.692   0.839   0.736   0.866   0.647
                              sin calibrar   0.011   0.010   0.097   0.007   0.115   0.035   0.033   0.012   0.033   0.017
        celda                 conformal      0.799   0.790   0.708   0.560   0.626   0.748   0.808   0.786   0.809   0.698
                              sin calibrar   0.027   0.052   0.086   0.031   0.091   0.049   0.027   0.041   0.060   0.024
C1E     campus × día × bloque conformal      0.782   0.774   0.677   0.672   0.698   0.788   0.846   0.752   0.885   0.690
                              sin calibrar   0.000   0.000   0.051   0.118   0.077   0.146   0.029   0.132   0.027   0.095
        celda                 conformal      0.792   0.770   0.702   0.653   0.695   0.810   0.833   0.839   0.844   0.798
                              sin calibrar   0.000   0.000   0.072   0.103   0.080   0.145   0.023   0.102   0.032   0.064
C1E-EUR campus × día × bloque conformal      0.802   0.728   0.732   0.649   0.646   0.738   0.781   0.712   0.850   0.668
                              sin calibrar   0.003   0.000   0.006   0.000   0.000   0.000   0.000   0.000   0.000   0.000
        celda                 conformal      0.809   0.756   0.732   0.628   0.680   0.750   0.795   0.768   0.811   0.703
                              sin calibrar   0.011   0.000   0.005   0.000   0.000   0.000   0.003   0.000   0.000   0.000
C1E-PER campus × día × bloque conformal      0.791   0.804   0.707   0.624   0.647   0.692   0.816   0.736   0.862   0.647
                              sin calibrar   0.011   0.010   0.022   0.007   0.010   0.035   0.000   0.012   0.000   0.017
        celda                 conformal      0.799   0.790   0.725   0.560   0.675   0.748   0.797   0.786   0.808   0.698
                              sin calibrar   0.027   0.052   0.015   0.031   0.011   0.049   0.009   0.041   0.015   0.024
C1E-TEN campus × día × bloque conformal      0.797   0.631   0.729   0.572   0.621   0.712   0.806   0.696   0.832   0.690
                              sin calibrar   0.003   0.000   0.006   0.000   0.000   0.000   0.000   0.000   0.000   0.000
        celda                 conformal      0.814   0.681   0.7

In [74]:
# Veredicto H5: cobertura conformal dentro de [75 %, 85 %] en la mayoría de los orígenes que deciden, en ambos granos
d = banda_val[(banda_val["USO"] == "decide") & (banda_val["METODO"] == "conformal")]
ok = d.assign(OK=d["COBERTURA"].between(0.75, 0.85)).groupby(["MODELO", "GRANO"])["OK"].sum().unstack()
resumen = banda_val[banda_val["USO"] == "decide"].groupby(["MODELO", "METODO"])[["COBERTURA", "ANCHO_MEDIO", "IS_REL"]].mean().round(3)
print(ok.assign(ADOPTA=(ok > sum(v == "decide" for v in ORIGENES.values()) / 2).all(axis=1)))
resumen

GRANO    campus × día × bloque  celda  ADOPTA
MODELO                                       
C1                           2      3   False
C1-EUR                       2      3   False
C1-PER                       2      3   False
C1-TEN                       2      3   False
C1E                          2      3   False
C1E-EUR                      3      3    True
C1E-PER                      2      3   False
C1E-TEN                      3      3    True


COBERTURA  ANCHO_MEDIO  IS_REL
MODELO  METODO                                      
C1      conformal         0.769       12.283   0.842
        sin calibrar      0.061        0.690   1.477
C1-EUR  conformal         0.761       12.503   0.878
        sin calibrar      0.030        0.331   1.604
C1-PER  conformal         0.763       12.444   0.874
        sin calibrar      0.012        0.000   1.644
C1-TEN  conformal         0.753       12.890   0.930
        sin calibrar      0.058        0.714   1.567
C1E     conformal         0.775       12.219   0.820
        sin calibrar      0.039        0.617   1.433
C1E-EUR conformal         0.764       12.436   0.866
        sin calibrar      0.003        0.000   1.648
C1E-PER conformal         0.763       12.444   0.874
        sin calibrar      0.012        0.000   1.644
C1E-TEN conformal         0.760       12.877   0.913
        sin calibrar      0.003        0.000   1.750

## 6 · Resultados

**6.1 · MASE (WAPE del modelo ÷ WAPE del naive) por origen y grano.** < 1 = le gana al naive.

In [75]:
pd.set_option("display.width", 220)
res.pivot_table(index=["MODELO", "GRANO"], columns="ORIGEN", values="MASE").round(3)

ORIGEN                        202410  202415  202420  202425  202520  202525  202610  202615  202620  202625
MODELO GRANO                                                                                                
C1     campus × día × bloque   1.000   1.000   0.970   0.922   0.981   0.708   0.946   0.762   0.866   0.808
       celda                   1.000   1.000   0.992   0.939   0.987   0.752   0.959   0.815   0.944   0.889
       unidad × bloque         1.000   1.000   0.926   0.649   0.923   0.561   0.930   0.727   0.826   0.766
C1-EUR campus × día × bloque   1.000   1.000   1.002   1.000   1.050   0.950   1.000   0.964   1.000   0.950
       celda                   1.000   1.000   1.016   1.000   1.047   0.963   1.000   0.967   1.000   0.968
...                              ...     ...     ...     ...     ...     ...     ...     ...     ...     ...
N1     celda                   1.015   1.092   1.007   1.103   1.018   1.015   1.045   0.984   1.036   1.089
       unidad × bloque         1.052   1.007   0.988   1.109   1.091   1.087   1.190   1.201   1.147   1.152
P1     campus × día × bloque   0.982   1.286   1.173   1.291   1.010   1.157   1.209   1.016   1.117   1.037
       celda                   1.041   1.343   1.160   1.169   1.131   1.213   1.275   1.016   1.163   0.987
       unidad × bloque         0.957   0.696   0.947   0.863   0.910   0.711   1.169   0.593   1.051   0.564

[66 rows x 10 columns]

In [76]:
# 6.2 · Veredicto con la regla de adopción (solo orígenes que deciden, granos celda y campus × día × bloque)
dec = res[(res["USO"] == "decide") & res["GRANO"].isin(["celda", "campus × día × bloque"]) & (res["MODELO"] != "N0")]
gana = dec.assign(GANA=dec["MASE"] < 1 - 1e-9).groupby(["MODELO", "GRANO"])["GANA"].sum().unstack()
N_DECIDE = sum(v == "decide" for v in ORIGENES.values())
veredicto = gana.assign(ADOPTA=(gana > N_DECIDE / 2).all(axis=1))
veredicto.join(dec.groupby("MODELO")["MASE"].agg(["mean", "std"]).round(3))

,campus × día × bloque,celda,ADOPTA,mean,std
MODELO,,,,,
C1,4,4,True,0.964,0.040
C1-EUR,0,0,False,1.011,0.020
C1-PER,1,0,False,1.007,0.016
C1-TEN,3,3,True,1.011,0.088
C1E,4,4,True,0.939,0.104
C1E-EUR,1,1,False,1.006,0.041
C1E-PER,1,0,False,1.007,0.016
C1E-TEN,2,2,False,1.060,0.147
C2,4,4,True,0.974,0.033


In [77]:
# 6.3 · Detalle por origen: error, sesgo, banda y punta a punta
cols = ["ORIGEN", "MODELO", "GRANO", "WAPE", "SESGO", "CORR", "TOP30", "COBERTURA_P10_P90", "ANCHO_MEDIO", "CRPS_REL",
        "GRILLA", "SECCIONES", "NRC_REALES", "SIN_HORARIO", "VERDAD_EXPLICADA_POR_PROYECCION", "W_NAIVE"]
res[res["GRANO"] == "campus × día × bloque"][[c for c in cols if c in res]].round(3)

,ORIGEN,MODELO,GRANO,WAPE,SESGO,CORR,TOP30,COBERTURA_P10_P90,ANCHO_MEDIO,CRPS_REL,GRILLA,SECCIONES,NRC_REALES,SIN_HORARIO,VERDAD_EXPLICADA_POR_PROYECCION,W_NAIVE
1,202410,N0,campus × día × bloque,0.114,-0.079,0.991,0.667,NaN,NaN,0.114,NaN,NaN,NaN,NaN,NaN,NaN
4,202410,N1,campus × día × bloque,0.125,-0.091,0.990,0.667,NaN,NaN,0.125,NaN,NaN,NaN,NaN,NaN,NaN
7,202410,P1,campus × día × bloque,0.112,-0.021,0.988,0.733,0.655,10.054,0.084,NaN,NaN,NaN,0.033,NaN,NaN
10,202410,G1,campus × día × bloque,0.139,-0.004,0.980,0.633,0.244,4.906,0.118,NaN,NaN,NaN,0.061,NaN,NaN
13,202410,G1b,campus × día × bloque,0.132,-0.016,0.983,0.667,0.238,4.789,0.111,NaN,NaN,NaN,0.061,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
646,202415,C1E-TEN,campus × día × bloque,0.304,0.186,0.933,0.667,0.000,0.000,0.304,NaN,NaN,NaN,NaN,NaN,1.0
649,202425,C1E-TEN,campus × día × bloque,0.311,0.103,0.901,0.800,0.000,0.000,0.311,NaN,NaN,NaN,NaN,NaN,1.0
652,202525,C1E-TEN,campus × día × bloque,0.257,0.184,0.949,0.733,0.000,0.000,0.257,NaN,NaN,NaN,NaN,NaN,1.0
655,202615,C1E-TEN,campus × día × bloque,0.321,0.114,0.849,0.467,0.000,0.000,0.321,NaN,NaN,NaN,NaN,NaN,1.0


**6.4 · Volumen proyectado (§4e).** WAPE de inscritos por tipo y estrato contra el real, sesgo total, error del número de secciones con el estándar real y, para Euribia, la cobertura P10–P90 por tipo. El error N con volumen real es el de H2 en §6.3.

In [78]:
vol_val.pivot_table(index="VARIANTE", columns="ORIGEN", values=["WAPE_TIPO", "SESGO_TOTAL", "ERROR_N_SEDE_COMP"]).round(3)

ERROR_N_SEDE_COMP                                                                 ... WAPE_TIPO                                                               
ORIGEN              202410 202415 202420 202425 202520 202525 202610 202615 202620 202625  ...    202410 202415 202420 202425 202520 202525 202610 202615 202620 202625
VARIANTE                                                                                   ...                                                                         
EUR                  0.146  0.132  0.168  0.125  0.279  0.230  0.162  0.426  0.157  0.427  ...     0.277  0.340  0.289  0.279  0.452  0.355  0.265  0.738  0.281  0.860
PER                  0.137  0.133  0.160  0.103  0.249  0.348  0.143  0.469  0.163  0.441  ...     0.265  0.321  0.287  0.288  0.439  0.463  0.264  0.813  0.297  0.931
TEN                  0.145  0.305  0.192  0.264  0.353  0.261  0.165  0.406  0.204  0.375  ...     0.274  0.493  0.289  0.431  0.480  0.420  0.284  0.783  0.328  0.861

[3 rows x 30 columns]

In [79]:
# 6.5 · Costo de proyectar el volumen: MASE de cada variante contra C1-oráculo, en los granos que deciden
c = res[res["MODELO"].str.startswith("C1") & res["GRANO"].isin(GRANOS_DECIDEN)].pivot_table(index=["ORIGEN", "GRANO"], columns="MODELO", values="MASE")
c.assign(**{f"COSTO_{m}": c[m] - c["C1"] for m in ("C1-EUR", "C1-PER", "C1-TEN")}).round(3)

MODELO                           C1  C1-EUR  C1-PER  C1-TEN    C1E  C1E-EUR  C1E-PER  C1E-TEN  COSTO_C1-EUR  COSTO_C1-PER  COSTO_C1-TEN
ORIGEN GRANO                                                                                                                           
202410 campus × día × bloque  1.000   1.000   1.000   1.000  1.128    0.920    1.000    0.892         0.000         0.000         0.000
       celda                  1.000   1.000   1.000   1.000  1.058    0.957    1.000    0.942         0.000         0.000         0.000
202415 campus × día × bloque  1.000   1.000   1.000   1.000  0.854    0.976    1.000    1.652         0.000         0.000         0.000
       celda                  1.000   1.000   1.000   1.000  0.909    1.009    1.000    1.529         0.000         0.000         0.000
202420 campus × día × bloque  0.970   1.002   0.993   0.904  0.972    1.016    0.993    0.868         0.031         0.023        -0.066
       celda                  0.992   1.016   1.010   0.971  0.999    1.018    1.010    0.946         0.024         0.018        -0.021
202425 campus × día × bloque  0.922   1.000   1.000   1.000  0.915    0.929    1.000    1.551         0.078         0.078         0.078
       celda                  0.939   1.000   1.000   1.000  0.912    0.950    1.000    1.344         0.061         0.061         0.061
202520 campus × día × bloque  0.981   1.050   1.036   1.181  0.898    1.054    1.036    1.296         0.069         0.055         0.200
       celda                  0.987   1.047   1.036   1.152  0.952    1.054    1.036    1.232         0.060         0.049         0.166
202525 campus × día × bloque  0.708   0.950   1.000   1.000  0.656    0.807    1.000    0.868         0.241         0.292         0.292
       celda                  0.752   0.963   1.000   1.000  0.703    0.854    1.000    0.895         0.211         0.248         0.248
202610 campus × día × bloque  0.946   1.000   1.000   0.978  0.904    1.007    1.000    1.093         0.054         0.054         0.032
       celda                  0.959   1.000   1.000   0.996  0.850    1.021    1.000    1.046         0.041         0.041         0.037
202615 campus × día × bloque  0.762   0.964   1.000   1.000  0.748    0.920    1.000    0.920         0.203         0.238         0.238
       celda                  0.815   0.967   1.000   1.000  0.774    0.931    1.000    0.929         0.152         0.185         0.185
202620 campus × día × bloque  0.866   1.000   1.000   0.935  0.812    1.012    1.000    1.167         0.134         0.134         0.069
       celda                  0.944   1.000   1.000   0.996  0.816    1.004    1.000    1.124         0.056         0.056         0.052
202625 campus × día × bloque  0.808   0.950   1.000   1.000  0.925    0.935    1.000    0.971         0.142         0.192         0.192
       celda                  0.889   0.968   1.000   1.000  0.882    0.958    1.000    0.975         0.079         0.111         0.111

## 7 · Largo plazo: pseudo-backtest por horizonte (27-10)

`Modelos.ipynb` proyecta 2028–2031 encadenando la matrícula de Ariadna a 2027. Aquí se repite esa mecánica desde bases antiguas y se compara con lo que realmente pasó. Diseño aprobado por Diego el 27-10.

**Pares base → objetivo:**
- **Bases:** 2022, 2023 y 2024 (sufijos 10 y 20 en pregrado; 15 y 25 en AV, solo informativo).
- **Objetivos:** el mismo sufijo, 1 a 4 años después (H+1 … H+4), hasta el último período real.
- Pares de pregrado: 6 en H+1 y H+2, 4 en H+3 y 2 en H+4.

**Sin mirar el futuro:**
- Las secciones, la clasificación de salas y β salen de la base o de antes. Si no hay pares previos para β (base 2022), se usa β = 1, crecimiento proporcional.
- **w es el de Modelos** (variante adoptada por nivel, último origen del banco). Se prueba el modelo tal como se usa, sin recalibrar.

**Volumen, cuatro variantes** (razón por tipo campus × materia × componente sobre los inscritos de la base):

| Variante | Razón base → T | Qué prueba |
|---|---|---|
| **ARI-REAL** | razón de 1 año de Modelos (UG: TEN; AV: EUR) × matrícula **real** anual de Ariadna del año T ÷ la del año base + 1, por carrera × sede (sede si la materia no es carrera) | la mecánica de Modelos si Ariadna acertara |
| PER | 1 | nada cambia |
| TEN | (crecimiento del campus en el año previo a la base)^h | tendencia compuesta (no sirve en la base 2022) |
| ORA | inscritos reales de T | piso: el error con volumen perfecto |

**Métricas por par, en celda y campus × día × bloque:**
- **MASE contra el naive de la base**, que copia el mapa de la base a cualquier distancia.
- **Banda:** residuos normalizados e = (real − punto) ÷ √(punto + 1). El factor **k_h** = Q80(|e| en H+h) ÷ Q80(|e| en H+1) dice cuánto hay que ensanchar la banda de 1 año. H+5 se extrapola con k_h = h^γ, ajustado en H+1 … H+4; si γ ≤ 0, k_5 = k_4.

**Modelo por horizonte:** en cada h, C1-ARI se queda solo si le gana al naive de la base (MASE < 1) en la mayoría de los pares de pregrado, en los dos granos. Si no, ese año usa el naive (w = 1). H+5 hereda la decisión de H+4. En AV (informativo) se elige el de menor MASE medio.

**Error del pronóstico de Ariadna (3b):** de su propio backtest (`DIAGNOSTICO_BACKTEST_28092026.csv`, rango oficial, el método que usa en cada nivel), se compara el error real H+1 … H+3 con su banda `n_bajo` / `n_alto` del mismo horizonte. Si la banda no cubre el error en el 80 % de los casos, se calcula un factor de ensanche para los escenarios bajo y alto. H+4 y H+5 heredan el de H+3.

**Salidas:** `VALIDACION_HORIZONTE.parquet` (métricas por par), `BANDA_HORIZONTE.parquet` (k_h, cobertura y modelo por nivel × h) y `ARIADNA_ERROR_HORIZONTE.parquet` (error, banda y factor de ensanche de Ariadna).

In [80]:
N_SIM_H = 30
H_MAX_MODELOS = 5                                     # 2031 queda a 5 años de la base 2026
PERIODOS_PL = set(pl["PERIODO"].unique())
ULTIMO = {s: max(p for p in PERIODOS_PL if p % 100 == s) for s in (10, 20, 15, 25)}
BASES_H = [a * 100 + s for a in (2022, 2023, 2024) for s in (10, 20, 15, 25)]
PARES_H = [(B, B + 100 * h, h) for B in BASES_H for h in range(1, 5) if B + 100 * h <= ULTIMO[B % 100]]
FUENTE_CORTO = {"UG": "TEN", "AV": "EUR"}

# Modelo adoptado y w, con la misma regla de Modelos §0
_v = res[res["MODELO"].isin(["C1-EUR", "C1-PER", "C1-TEN"]) & res["GRANO"].isin(GRANOS_DECIDEN)].assign(NIVEL=lambda d: d["ORIGEN"].map(nivel_de))
_d = _v[_v["USO"] == "decide"]
_g = _d.assign(G=_d["MASE"] < 1 - 1e-9).groupby(["MODELO", "GRANO"])["G"].sum().unstack()
_pasa = _g[(_g > _d["ORIGEN"].nunique() / 2).all(axis=1)].index
ADOPTADO = {"UG": _d[_d["MODELO"].isin(_pasa)].groupby("MODELO")["MASE"].mean().idxmin() if len(_pasa) else None,
            "AV": _v[_v["NIVEL"] == "AV"].groupby("MODELO")["MASE"].mean().idxmin()}
W_LP = {n: (float(res[(res["MODELO"] == m) & (res["ORIGEN"].map(nivel_de) == n)].sort_values("ORIGEN")["W_NAIVE"].iloc[-1]) if m else 1.0)
        for n, m in ADOPTADO.items()}
W_LP_E = {n: (float(res[(res["MODELO"] == m.replace("C1-", "C1E-")) & (res["ORIGEN"].map(nivel_de) == n)].sort_values("ORIGEN")["W_NAIVE"].iloc[-1]) if m else 1.0)
          for n, m in ADOPTADO.items()}
print("modelo adoptado:", ADOPTADO, "· w:", W_LP, "· w con naive escalado (D19):", W_LP_E, "· pares:", len(PARES_H))

ari_anio = ariadna.assign(ANIO=ariadna["periodo"] // 100)
mat_anual = ari_anio.groupby(["ANIO", "cod_carrera", "sede"])["n"].sum()
mat_sede = ari_anio.groupby(["ANIO", "sede"])["n"].sum()


def tipos_base(B):
    s, _ = sf(B, B + 1)
    return s.groupby(TIPO_V)["VOL"].sum().index


def razon_en_tipos(razon, idx):
    df = idx.to_frame(index=False)
    return pd.Series(aplicar_razon(df, razon)["RAZON"].to_numpy(), index=idx)


def razon_corto(B, nivel):
    # razón de 1 año que usa Modelos (UG: TEN; AV: EUR); sin historia previa, persistencia
    if FUENTE_CORTO[nivel] == "TEN":
        return razon_tendencia(B + 100) if (B - 100) in set(ph["PERIODO"]) else None
    return razon_euribia(B + 100)


def razon_ari_real(B, T, idx):
    nivel = nivel_de(B)
    rc = razon_corto(B, nivel)
    r1 = razon_en_tipos(rc, idx) if rc is not None else pd.Series(1.0, index=idx)
    a0, a1 = B // 100 + 1, T // 100
    mat, camp = idx.get_level_values("MATERIA"), idx.get_level_values("COD_CAMPUS")
    km = pd.MultiIndex.from_arrays([mat, camp])
    r_m = mat_anual.loc[a1].reindex(km).to_numpy() / mat_anual.loc[a0].reindex(km).to_numpy()
    r_s = mat_sede.loc[a1].reindex(camp).to_numpy() / mat_sede.loc[a0].reindex(camp).to_numpy()
    ok_m = np.isfinite(r_m) & (r_m > 0)
    crec = np.where(ok_m, r_m, np.where(np.isfinite(r_s) & (r_s > 0), r_s, 1.0))
    return (r1 * crec).clip(0.25, 4.0).rename("RAZON"), float(ok_m.mean())


def razon_ten_compuesta(B, T):
    if (B - 100) not in set(ph["PERIODO"]):
        return None
    return (razon_tendencia(B + 100) ** ((T - B) // 100)).clip(0.25, 4.0)

modelo adoptado: {'UG': 'C1-TEN', 'AV': 'C1-EUR'} · w: {'UG': 0.8, 'AV': 0.0} · w con naive escalado (D19): {'UG': 1.0, 'AV': 1.0} · pares: 36


In [81]:
filas_h, RES_H, BETA_H = [], {}, {}
for B, T, h in PARES_H:
    nivel = nivel_de(B)
    uso = "decide" if nivel == "UG" else "informativo"
    rc_, _ = real(T)
    rv = {"celda": rc_, "campus × día × bloque": agr_cdb(rc_)}
    nb_, _ = real(B)
    naive = {"celda": nb_, "campus × día × bloque": agr_cdb(nb_)}
    if B not in BETA_H:
        try:
            BETA_H[B] = elasticidad(B + 100)[0]
        except ValueError:
            BETA_H[B] = 1.0                           # base 2022: sin pares previos para calibrar β
    beta = BETA_H[B]
    idx = tipos_base(B)
    r_ari, cob_ari = razon_ari_real(B, T, idx)
    variantes = {"ARI-REAL": r_ari, "PER": pd.Series(1.0, index=idx, name="RAZON"), "TEN": razon_ten_compuesta(B, T), "ORA": "oraculo"}
    w = W_LP[nivel]
    for nombre_v, rz in variantes.items():
        if rz is None:
            continue
        gc, *_ = generar_g1(T, n_sim=N_SIM_H, beta=beta, razon=None if nombre_v == "ORA" else rz, base=B)
        g_cel = vec_crece(G1_CRECE["ultimo"])
        naive_e = {"celda": nb_ * g_cel, "campus × día × bloque": agr_cdb(nb_ * g_cel)}
        for g in GRANOS_DECIDEN:
            sims = gc if g == "celda" else agr_cdb(gc)
            for pref, w_, nv in (("C1", w, naive[g]), ("C1E", W_LP_E[nivel], naive_e[g])):
                punto = w_ * nv + (1 - w_) * sims.mean(0)
                fila = {"BASE": B, "OBJETIVO": T, "H": h, "NIVEL": nivel, "USO": uso, "GRANO": g, "MODELO": f"{pref}-{nombre_v}",
                        "W_NAIVE": w_, "BETA": beta, "WAPE": wape(rv[g], punto), "WAPE_NAIVE_BASE": wape(rv[g], naive[g]),
                        "SESGO": punto.sum() / rv[g].sum() - 1, "COBERTURA_MATRICULA_CARRERA": cob_ari if nombre_v == "ARI-REAL" else np.nan}
                filas_h.append(fila)
                RES_H[(f"{pref}-{nombre_v}", B, T, g)] = residuos(rv[g], punto, g)
    for g in GRANOS_DECIDEN:
        filas_h.append({"BASE": B, "OBJETIVO": T, "H": h, "NIVEL": nivel, "USO": uso, "GRANO": g, "MODELO": "NAIVE_BASE",
                        "W_NAIVE": 1.0, "WAPE": wape(rv[g], naive[g]), "WAPE_NAIVE_BASE": wape(rv[g], naive[g]),
                        "SESGO": naive[g].sum() / rv[g].sum() - 1})
        RES_H[("NAIVE_BASE", B, T, g)] = residuos(rv[g], naive[g], g)
    print(f"{B} -> {T} (H+{h}) listo")

val_h = pd.DataFrame(filas_h)
val_h["MASE"] = val_h["WAPE"] / val_h["WAPE_NAIVE_BASE"]
val_h.to_parquet(BBDD / "VALIDACION_HORIZONTE.parquet", index=False)
val_h[val_h["GRANO"] == "celda"].pivot_table(index=["NIVEL", "MODELO"], columns="H", values="MASE", aggfunc="mean").round(3)

C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:70: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  _u = base[base["EN_BASE"].fillna(False).astype(bool)].drop_duplicates("EST")
C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:70: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  _u = base[base["EN_BASE"].fillna(False).astype(bool)].drop_duplicates("EST")
C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:66: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future v

202210 -> 202310 (H+1) listo


C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:70: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  _u = base[base["EN_BASE"].fillna(False).astype(bool)].drop_duplicates("EST")
C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:70: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  _u = base[base["EN_BASE"].fillna(False).astype(bool)].drop_duplicates("EST")
C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:66: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future v

202210 -> 202410 (H+2) listo


C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:70: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  _u = base[base["EN_BASE"].fillna(False).astype(bool)].drop_duplicates("EST")
C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:70: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  _u = base[base["EN_BASE"].fillna(False).astype(bool)].drop_duplicates("EST")
C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:66: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future v

202210 -> 202510 (H+3) listo


... (53 outputs left out)

202425 -> 202525 (H+1) listo


C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:70: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  _u = base[base["EN_BASE"].fillna(False).astype(bool)].drop_duplicates("EST")
C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:70: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  _u = base[base["EN_BASE"].fillna(False).astype(bool)].drop_duplicates("EST")
C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:70: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change 

202425 -> 202625 (H+2) listo


H                       1      2      3      4
NIVEL MODELO                                  
AV    C1-ARI-REAL   0.967  0.799  0.754  1.083
      C1-ORA        0.889  0.805  0.736  0.635
      C1-PER        1.011  1.010  1.007  1.007
      C1-TEN        1.189  1.402  2.333    NaN
      C1E-ARI-REAL  0.951  0.816  0.766  0.945
      C1E-ORA       0.838  0.748  0.668  0.536
      C1E-PER       1.000  1.000  1.000  1.000
      C1E-TEN       1.177  1.384  2.309    NaN
      NAIVE_BASE    1.000  1.000  1.000  1.000
UG    C1-ARI-REAL   1.009  1.001  0.995  0.993
      C1-ORA        0.991  0.977  0.973  0.972
      C1-PER        1.007  1.005  1.007  1.003
      C1-TEN        1.009  1.030  1.017    NaN
      C1E-ARI-REAL  1.053  1.012  0.959  0.962
      C1E-ORA       0.956  0.917  1.006  1.024
      C1E-PER       1.000  1.000  1.000  1.000
      C1E-TEN       1.080  1.264  1.218    NaN
      NAIVE_BASE    1.000  1.000  1.000  1.000

In [82]:
# Modelo por horizonte y factor de ensanche k_h de la banda
filas_k = []
for nivel in ("UG", "AV"):
    d = val_h[(val_h["NIVEL"] == nivel) & (val_h["MODELO"] == "C1-ARI-REAL")]
    decision = {}
    for h in sorted(d["H"].unique()):
        x = d[d["H"] == h]
        if nivel == "UG":
            gana = x.assign(G=x["MASE"] < 1 - 1e-9).groupby("GRANO")["G"].sum()
            decision[h] = "C1" if (gana > x["BASE"].nunique() / 2).all() else "NAIVE"
        else:
            decision[h] = "C1" if x["MASE"].mean() < 1 - 1e-9 else "NAIVE"
    decision[H_MAX_MODELOS] = decision[max(decision)]
    for g in GRANOS_DECIDEN:
        for modelo_h in ("C1", "C1E", "NAIVE"):
            clave = {"C1": "C1-ARI-REAL", "C1E": "C1E-ARI-REAL", "NAIVE": "NAIVE_BASE"}[modelo_h]
            pares = [(B, T, h) for B, T, h in PARES_H if nivel_de(B) == nivel]
            e_h = {h: np.abs(pd.concat([RES_H[(clave, B, T, g)] for B, T, hh in pares if hh == h])["E"]) for h in sorted({p[2] for p in pares})}
            ref = np.quantile(e_h[1], 0.8)
            k = {h: float(np.quantile(e, 0.8) / ref) for h, e in e_h.items()}
            hs = np.array([h for h in k if h > 1]); ks = np.array([k[h] for h in hs])
            gamma = float((np.log(hs) * np.log(ks)).sum() / (np.log(hs) ** 2).sum())
            k[H_MAX_MODELOS] = H_MAX_MODELOS ** gamma if gamma > 0 else k[max(hs)]
            # Cobertura P10–P90 dentro del backtest: cuantiles de H+1 por clase × tramo, sin y con k_h (alrededor de la mediana)
            q1, glob1 = cuantiles(pd.concat([RES_H[(clave, B, T, g)] for B, T, h in pares if h == 1]))
            for h in sorted(k):
                fila = {"NIVEL": nivel, "GRANO": g, "MODELO_BANDA": modelo_h, "H": h, "K": k[h], "GAMMA": gamma,
                        "K_ORIGEN": "medido" if h in e_h else "extrapolado", "MODELO_H": decision[h],
                        "N_PARES": sum(1 for p in pares if p[2] == h)}
                if h in e_h:
                    r = pd.concat([RES_H[(clave, B, T, g)] for B, T, hh in pares if hh == h])
                    i10, i50, i90 = (int(np.argmin(np.abs(QS - a))) for a in (0.10, 0.50, 0.90))
                    Q = np.stack([q1.get((c, t), glob1) for c, t in zip(r["CLASE"], r["TRAMO"])])
                    for etiqueta, kk in (("COB_SIN_K", 1.0), ("COB_CON_K", k[h])):
                        lo = Q[:, i50] + kk * (Q[:, i10] - Q[:, i50]); hi = Q[:, i50] + kk * (Q[:, i90] - Q[:, i50])
                        fila[etiqueta] = float(((r["E"] >= lo) & (r["E"] <= hi)).mean())
                filas_k.append(fila)
banda_h = pd.DataFrame(filas_k)
banda_h.to_parquet(BBDD / "BANDA_HORIZONTE.parquet", index=False)
banda_h.pivot_table(index=["NIVEL", "GRANO", "MODELO_BANDA"], columns="H", values=["K", "COB_SIN_K", "COB_CON_K"]).round(2)

COB_CON_K                   COB_SIN_K                      K                        
H                                                1     2     3     4         1     2     3     4    1     2     3     4     5
NIVEL GRANO                 MODELO_BANDA                                                                                     
AV    campus × día × bloque C1                0.80  0.79  0.82  0.81      0.80  0.69  0.60  0.50  1.0  1.30  1.62  2.41  2.37
                            C1E               0.80  0.81  0.82  0.83      0.80  0.68  0.62  0.50  1.0  1.39  1.65  2.18  2.29
                            NAIVE             0.80  0.81  0.80  0.84      0.80  0.66  0.59  0.46  1.0  1.48  1.75  2.29  2.48
      celda                 C1                0.80  0.80  0.81  0.82      0.80  0.73  0.66  0.54  1.0  1.20  1.39  1.84  1.81
                            C1E               0.80  0.81  0.82  0.81      0.80  0.73  0.66  0.58  1.0  1.26  1.43  1.67  1.76
                            NAIVE             0.80  0.81  0.85  0.86      0.80  0.70  0.63  0.56  1.0  1.33  1.59  1.80  1.97
UG    campus × día × bloque C1                0.80  0.79  0.81  0.87      0.80  0.68  0.59  0.52  1.0  1.31  1.66  2.25  2.30
                            C1E               0.80  0.80  0.83  0.86      0.80  0.70  0.64  0.56  1.0  1.29  1.60  2.01  2.10
                            NAIVE             0.80  0.78  0.81  0.87      0.80  0.67  0.58  0.51  1.0  1.32  1.72  2.32  2.39
      celda                 C1                0.80  0.83  0.83  0.88      0.80  0.69  0.62  0.55  1.0  1.34  1.51  1.83  1.94
                            C1E               0.80  0.83  0.83  0.87      0.80  0.70  0.64  0.56  1.0  1.31  1.44  1.72  1.82
                            NAIVE             0.81  0.83  0.84  0.89      0.81  0.69  0.62  0.55  1.0  1.35  1.56  1.88  2.02

In [83]:
# 3b · Error del pronóstico de Ariadna contra su banda bajo/alto, por horizonte
ari_full = pd.read_parquet(DESCARGAS / "MODELO_PROYECCCIÓN_SIMPLE" / "PROYECCION_MATRICULA_2026_2031_28092026.parquet",
                           columns=["periodo", "nivel_academico", "n", "n_bajo", "n_alto", "tipo", "metodo"])
ari_full["ANIO"] = ari_full["periodo"].astype(int) // 100
proy = ari_full[ari_full["tipo"] == "PROYECTADO"]
metodo_ari = proy.groupby(["nivel_academico", "metodo"])["n"].sum().reset_index().sort_values("n").drop_duplicates("nivel_academico", keep="last").set_index("nivel_academico")["metodo"]
agr = proy.groupby(["nivel_academico", "ANIO"])[["n", "n_bajo", "n_alto"]].sum()
agr["H"] = agr.index.get_level_values("ANIO") - 2026
agr["MEDIA_BAJO"] = 1 - agr["n_bajo"] / agr["n"]; agr["MEDIA_ALTO"] = agr["n_alto"] / agr["n"] - 1
bt = pd.read_csv(DESCARGAS / "MODELO_PROYECCCIÓN_SIMPLE" / "DIAGNOSTICO_BACKTEST_28092026.csv", encoding="utf-8-sig")
bt = bt[(bt["rango"] == "oficial") & (bt["metodo"] == bt["nivel_academico"].map(metodo_ari))].copy()
bt["ERR"] = bt["proyeccion"] / bt["real"] - 1
filas_a = []
for (nivel, h), x in bt.groupby(["nivel_academico", "h"]):
    a = agr.loc[nivel]; a = a[a["H"] == h].iloc[0]
    # el real queda dentro si −MEDIA_ALTO ≤ ERR ≤ MEDIA_BAJO (proyección − real relativo)
    dentro = ((x["ERR"] <= a["MEDIA_BAJO"] / (1 - a["MEDIA_BAJO"])) & (x["ERR"] >= -a["MEDIA_ALTO"] / (1 + a["MEDIA_ALTO"]))).mean()
    q80 = np.quantile(np.abs(x["ERR"]), 0.8)
    filas_a.append({"NIVEL": nivel, "H": h, "METODO": metodo_ari[nivel], "N_PUNTOS": len(x), "ERR_MEDIO": x["ERR"].mean(),
                    "ERR_ABS_Q80": q80, "MEDIA_BANDA": (a["MEDIA_BAJO"] + a["MEDIA_ALTO"]) / 2, "COBERTURA_BANDA": dentro,
                    "ENSANCHE": max(1.0, q80 / ((a["MEDIA_BAJO"] + a["MEDIA_ALTO"]) / 2))})
ari_err = pd.DataFrame(filas_a)
ari_err.to_parquet(BBDD / "ARIADNA_ERROR_HORIZONTE.parquet", index=False)
ari_err.round(3)

,NIVEL,H,METODO,N_PUNTOS,ERR_MEDIO,ERR_ABS_Q80,MEDIA_BANDA,COBERTURA_BANDA,ENSANCHE
0,AV,1,HOLT,5,0.032,0.117,0.117,0.800,1.000
1,AV,2,HOLT,5,-0.035,0.237,0.165,0.800,1.436
2,AV,3,HOLT,4,-0.053,0.307,0.202,0.750,1.521
3,MG,1,REG_LINEAL_QUIEBRE,4,0.120,0.245,0.185,0.750,1.330
4,MG,2,REG_LINEAL_QUIEBRE,4,0.328,0.616,0.226,0.500,2.728
5,MG,3,REG_LINEAL_QUIEBRE,3,0.644,0.905,0.261,0.333,3.469
6,PH,1,REG_LINEAL,5,-0.098,0.180,0.232,0.800,1.000
7,PH,2,REG_LINEAL,5,-0.168,0.268,0.323,0.600,1.000
8,PH,3,REG_LINEAL,4,-0.262,0.282,0.396,0.750,1.000
9,UG,1,MA_TASAS,5,-0.008,0.048,0.043,0.800,1.130


**7.1 · Lectura rápida.** MASE medio por horizonte en los dos granos (< 1 = le gana al naive de la base), y el costo del volumen: ARI-REAL contra ORA.

In [84]:
t = val_h[val_h["USO"] == "decide"].pivot_table(index=["GRANO", "MODELO"], columns="H", values="MASE", aggfunc="mean")
display(t.round(3))
val_h[(val_h["USO"] == "decide") & (val_h["MODELO"] == "C1-ARI-REAL")].pivot_table(index=["GRANO", "BASE"], columns="H", values="MASE").round(3)

H                                       1      2      3      4
GRANO                 MODELO                                  
campus × día × bloque C1-ARI-REAL   1.006  1.001  0.987  0.987
                      C1-ORA        0.987  0.964  0.962  0.954
                      C1-PER        1.006  1.008  1.007  1.003
                      C1-TEN        1.005  1.044  1.019    NaN
                      C1E-ARI-REAL  1.056  1.015  0.931  0.943
                      C1E-ORA       0.943  0.890  1.030  1.034
                      C1E-PER       1.000  1.000  1.000  1.000
                      C1E-TEN       1.083  1.357  1.271    NaN
                      NAIVE_BASE    1.000  1.000  1.000  1.000
celda                 C1-ARI-REAL   1.009  1.001  0.995  0.993
                      C1-ORA        0.991  0.977  0.973  0.972
                      C1-PER        1.007  1.005  1.007  1.003
                      C1-TEN        1.009  1.030  1.017    NaN
                      C1E-ARI-REAL  1.053  1.012  0.959  0.962
                      C1E-ORA       0.956  0.917  1.006  1.024
                      C1E-PER       1.000  1.000  1.000  1.000
                      C1E-TEN       1.080  1.264  1.218    NaN
                      NAIVE_BASE    1.000  1.000  1.000  1.000

H                                 1      2      3      4
GRANO                 BASE                              
campus × día × bloque 202210  0.999  0.969  0.974  0.980
                      202220  1.018  0.991  0.984  0.994
                      202310  0.949  0.989  1.001    NaN
                      202320  0.952  0.978  0.989    NaN
                      202410  1.066  1.045    NaN    NaN
                      202420  1.054  1.037    NaN    NaN
celda                 202210  1.003  0.976  0.976  0.987
                      202220  1.015  0.997  0.994  0.999
                      202310  0.968  0.999  1.008    NaN
                      202320  0.977  0.987  1.003    NaN
                      202410  1.047  1.032    NaN    NaN
                      202420  1.046  1.016    NaN    NaN

## 8 · Rango de capacidad: validación informativa (27-10)

`Modelos.ipynb` reparte la demanda de cada celda por el **rango de sala que necesitan** las secciones: 0-20, 21-40, 41-60 y 60 o más, los cortes de la planta. Aquí se mide, en los mismos orígenes de 1 año, si ese reparto acierta mejor que el naive.

**Reglas, iguales a las de Modelos:**
- **Tolerancia por clase:** percentil 90 de inscritos ÷ capacidad por sala-bloque en el período base (T−100), con piso 1.
- **Capacidad que necesita la sección:** inscritos base × razón del estrato^(1 − β), dividido por el mayor entre la tolerancia y el sobrecupo que ya tenía en su sala. G1b agrega secciones al ritmo razón^β, así que el resto del crecimiento engorda las secciones; el sobrecupo histórico no cuenta como falta.
- **C1 por rango:** w · naive por rango (los tamaños de la base) + (1 − w) · G1b por rango, con la variante adoptada por nivel (UG: TEN; AV: EUR) y su w.
- **Verdad por rango:** el mapa real de T, repartido por los tamaños reales de sus secciones.

**Métrica:** WAPE y MASE contra el naive en los granos celda × rango y campus × día × bloque × rango. Es **informativa**: el rango desagrega el modelo ya adoptado y no pasa por la regla de adopción.

**Salida:** `VALIDACION_RANGO.parquet`.

In [85]:
RANGOS = ["0-20", "21-40", "41-60", "60 o más"]
TOPES = [20, 40, 60]
NR = len(RANGOS)
rango_de = lambda x: np.digitize(np.asarray(x, dtype=float), TOPES, right=True)
_oferta_cap = oferta.set_index(["EDIFICIO", "SALA"])["CAPACIDAD"]


def tam_necesario(ins_proy, ins_base, cap_sec, tol):
    # igual que Modelos: inscritos proyectados ÷ max(tolerancia, sobrecupo que ya tenía la sección en su sala)
    sobre = np.where(np.asarray(cap_sec, dtype=float) > 0, np.asarray(ins_base, dtype=float) / np.asarray(cap_sec, dtype=float), 0.0)
    return np.asarray(ins_proy, dtype=float) / np.maximum(np.asarray(tol, dtype=float), np.nan_to_num(sobre))


def tolerancia(secs, filas_):
    x = filas_.merge(secs[["SEC", "INSCRITOS"]], on="SEC")
    sb = x.groupby(["EDIFICIO", "SALA", "DIA", "BLOQUE", "CLASE"])["INSCRITOS"].sum().reset_index()
    sb["CAP"] = _oferta_cap.reindex(pd.MultiIndex.from_frame(sb[["EDIFICIO", "SALA"]])).to_numpy()
    sb = sb[sb["CAP"] > 0]
    return (sb["INSCRITOS"] / sb["CAP"]).groupby(sb["CLASE"]).quantile(0.9).clip(lower=1.0).to_dict()


def reparto_real(P, clase, tol):
    # mapa real de P repartido por el rango que necesitan sus secciones (tamaños reales de P)
    secs, filas_ = secciones_fisicas(P, clase)
    f = filas_.merge(secs[["SEC", "INSCRITOS"]], on="SEC")
    c = filas_[["SEC", "EDIFICIO", "SALA"]].drop_duplicates()
    c["CAP"] = _oferta_cap.reindex(pd.MultiIndex.from_frame(c[["EDIFICIO", "SALA"]])).to_numpy()
    rr = rango_de(tam_necesario(f["INSCRITOS"], f["INSCRITOS"], f["SEC"].map(c.groupby("SEC")["CAP"].max()), f["CLASE"].map(tol).fillna(1.0)))
    cel = f["CAMP"].map(CAMP).to_numpy() * N_SC + resto(f["DIA"].map(DIA_I).to_numpy(), f["BLOQUE"].to_numpy(), f["CLASE"].map(CLA_I).to_numpy())
    m = np.bincount(cel * NR + rr, weights=f["PESO"].to_numpy(), minlength=N_CELDAS * NR).reshape(N_CELDAS, NR)
    tot = m.sum(1, keepdims=True)
    return np.where(tot > 0, m / np.where(tot > 0, tot, 1), np.eye(NR)[0])


agr_cdb_r = lambda m: m.reshape(len(CAMP), len(DIAS), NB, len(CLASES), NR).sum(3).reshape(-1)
filas_rg = []
for T, uso in ORIGENES.items():
    nivel = nivel_de(T)
    modelo = ADOPTADO[nivel] or "C1-PER"
    fuente = modelo.split("-")[1]
    w = float(res[(res["MODELO"] == modelo) & (res["ORIGEN"] == T)]["W_NAIVE"].iat[0])
    beta = float(res[(res["MODELO"] == "G1b") & (res["ORIGEN"] == T)]["BETA"].iat[0])
    clase_prev = clasificar([p for p in pl["PERIODO"].unique() if T - 200 <= p < T])
    tol = tolerancia(*secciones_fisicas(T - 100, clase_prev))
    generar_g1(T, n_sim=50, beta=beta, razon=razones(T)[fuente], tol=tol)
    g_r = RANGO_G1.pop("ultimo").mean(0).reshape(N_CELDAS, NR)
    real_r = real(T)[0][:, None] * reparto_real(T, clasificar([p for p in pl["PERIODO"].unique() if T - 200 <= p <= T]), tol)
    naive_r = real(T - 100)[0][:, None] * reparto_real(T - 100, clase_prev, tol)
    pred_r = w * naive_r + (1 - w) * g_r
    for grano, f_ in (("celda × rango", lambda m: m.reshape(-1)), ("campus × día × bloque × rango", agr_cdb_r)):
        rv, pv, nv = f_(real_r), f_(pred_r), f_(naive_r)
        filas_rg.append({"ORIGEN": T, "USO": uso, "MODELO": modelo, "GRANO": grano, "W_NAIVE": w, "BETA": beta,
                         "TOL_SALA": tol.get("SALA", np.nan), "TOL_LAB": tol.get("LABORATORIO", np.nan),
                         "WAPE": wape(rv, pv), "WAPE_NAIVE": wape(rv, nv), "MASE": wape(rv, pv) / wape(rv, nv),
                         **{f"PARTE_REAL_{RANGOS[i]}": float(real_r[:, i].sum() / real_r.sum()) for i in range(NR)},
                         **{f"PARTE_PRED_{RANGOS[i]}": float(pred_r[:, i].sum() / pred_r.sum()) for i in range(NR)}})
    print(f"{T} rango listo")
val_rango = pd.DataFrame(filas_rg)
val_rango.to_parquet(BBDD / "VALIDACION_RANGO.parquet", index=False)
display(val_rango.pivot_table(index="GRANO", columns="ORIGEN", values="MASE").round(3))
val_rango[val_rango["GRANO"] == "celda × rango"][["ORIGEN", "TOL_SALA", "TOL_LAB"] + [c for c in val_rango if c.startswith("PARTE_")]].round(3)

C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:70: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  _u = base[base["EN_BASE"].fillna(False).astype(bool)].drop_duplicates("EST")


202410 rango listo


C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:70: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  _u = base[base["EN_BASE"].fillna(False).astype(bool)].drop_duplicates("EST")


202420 rango listo


C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:70: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  _u = base[base["EN_BASE"].fillna(False).astype(bool)].drop_duplicates("EST")


202520 rango listo


... (12 outputs left out)

C:\Users\usuario\AppData\Local\Temp\ipykernel_52592\4101355015.py:70: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  _u = base[base["EN_BASE"].fillna(False).astype(bool)].drop_duplicates("EST")


202625 rango listo


ORIGEN,202410,202415,202420,202425,202520,202525,202610,202615,202620,202625
GRANO,,,,,,,,,,
campus × día × bloque × rango,1.0,1.0,1.009,1.0,1.089,0.968,0.997,0.978,1.005,1.012
celda × rango,1.0,1.0,1.011,1.0,1.072,0.971,0.997,0.991,1.010,1.033


,ORIGEN,TOL_SALA,TOL_LAB,PARTE_REAL_0-20,PARTE_REAL_21-40,PARTE_REAL_41-60,PARTE_REAL_60 o más,PARTE_PRED_0-20,PARTE_PRED_21-40,PARTE_PRED_41-60,PARTE_PRED_60 o más
0,202410,1.091,1.636,0.500,0.248,0.156,0.096,0.494,0.234,0.168,0.105
2,202420,1.000,1.429,0.467,0.273,0.139,0.121,0.479,0.254,0.133,0.134
4,202520,1.000,1.201,0.399,0.311,0.134,0.156,0.427,0.309,0.138,0.126
6,202610,1.073,1.750,0.462,0.241,0.160,0.136,0.477,0.245,0.142,0.136
8,202620,1.000,1.200,0.397,0.301,0.158,0.144,0.399,0.311,0.132,0.157
10,202415,1.000,1.000,0.657,0.266,0.071,0.006,0.668,0.252,0.077,0.003
12,202425,1.000,1.120,0.701,0.237,0.057,0.004,0.688,0.246,0.062,0.004
14,202525,1.000,1.000,0.654,0.254,0.085,0.007,0.694,0.243,0.058,0.004
16,202615,1.000,1.000,0.601,0.305,0.094,0.000,0.628,0.290,0.079,0.004
18,202625,1.000,1.000,0.640,0.279,0.076,0.005,0.631,0.269,0.088,0.013
